## Data Annotation (GENCODE v49)

Gene annotations were obtained from GENCODE (release v49), downloaded from the official repository (https://www.gencodegenes.org) \
and used to map Ensembl gene identifiers to gene symbols and genomic coordinates. Only protein-coding genes were retained for downstream analysis.

References:
- Harrow et al., 2012
- Frankish et al., 2021


# Downloads and processes TCGA data:
- Mutation (MAF)
- Gene Expression
- Copy Number Alteration (CNA)
- miRNA expression

Outputs:
- Gene-level normalized features per cancer type


# Data acquisition

Cohort selection used for large-scale multi-omics data processing from the Genomic Data Commons (GDC).

In [ ]:
import json, requests, gzip, re, os
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm import tqdm
import gseapy as gp
from tqdm.auto import tqdm
print(os.getcwd())
from collections import defaultdict
# from tqdm.notebook import tqdm

OUT_DIR = Path("../data/gdc_cache")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data/tcga_maf")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("data/tcga_omics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8

CANCERS = ['TCGA-BLCA', 'TCGA-BRCA', 'TCGA-COAD', 'TCGA-ESCA', 'TCGA-LUAD', 'TCGA-LUSC', 'TCGA-STAD']


FILE_LIMITS = {
    "mutation": 30,
    "expression": 30,
    "cna": 30,
    "mirna": 15,
}

COLORS = [
    '#0077B6','#0000FF','#00B4D8','#48EAC4','#F1C0E8','#B9FBC0',
    '#32CD32','#BEE1E6','#8A2BE2','#E377C2','#8EECF5','#A3C4F3',
    '#FFB347','#FFD700','#FF69B4','#CD5C5C','#7FFFD4','#FF7F50',
    '#C71585','#20B2AA','#6A5ACD','#40E0D0','#FF8C00','#DC143C',
    '#9ACD32','#1F77B4','#FF1493','#2E8B57','#D2691E','#9932CC',
    '#00CED1','#FF4500','#708090'
]


# Data sources and configuration

Define the data modalities, external resources, and annotation files.

In [ ]:
DATA_TYPES = {
    "mutation": "Masked Somatic Mutation",
    "expression": "Gene Expression Quantification",
    "cna": "Copy Number Segment",
    "mirna": "miRNA Expression Quantification",
}

GDC_FILES_API = "https://api.gdc.cancer.gov/files"
GDC_DATA_API = "https://api.gdc.cancer.gov/data"

# GENCODE_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz"

def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"

GENCODE_URL, GENCODE_VERSION = get_latest_gencode_url()
# GTF_PATH = OUT_DIR / "gencode.v38.annotation.gtf.gz"
GTF_PATH = OUT_DIR / f"gencode.{GENCODE_VERSION}.annotation.gtf.gz"

MIRTARBASE_FILES = {
    "SE_WR": ("../data/miRTarBase_SE_WR.csv", 3.0),  
    "SE_W":  ("../data/miRTarBase_SE_W.csv", 2.0),
    "SE_R":  ("../data/miRTarBase_SE_R.csv", 2.0),
    "WE_CLIP": ("../data/miRTarBase_WE_Clip.csv", 1.0),
    "WE_OTHER": ("../data/miRTarBase_WE_Other.csv", 1.0),
}

# GENCODE GTF

Retrieving and parsing of gene annotation data to map genomic coordinates, harmonizing gene identifiers, and enabling gene-level integration across multiple omics modalities.

In [ ]:
def download_gtf():
    if GTF_PATH.exists() and GTF_PATH.stat().st_size > 1e6:
        print("[cache] GTF")
        return GTF_PATH
    print("[download] GTF")
    r = requests.get(GENCODE_URL, stream=True)
    r.raise_for_status()
    with open(GTF_PATH, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    return GTF_PATH

# def download_gtf():
#     if GTF_PATH.exists() and GTF_PATH.stat().st_size > 1e7:
#         print(f"[cache] GTF (assumed {GENCODE_VERSION})")
#         return GTF_PATH

def parse_gtf():
    path = download_gtf()

    gene_map = {}
    gene_annot = {}

    # Count total lines for tqdm
    with gzip.open(path, "rt") as f:
        total_lines = sum(1 for _ in f)

    with gzip.open(path, "rt") as f:
        for line in tqdm(f, total=total_lines, desc="Parsing GTF"):
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if parts[2] != "gene":
                continue

            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            attr = parts[8]

            gid = re.search(r'gene_id "([^"]+)"', attr)
            gname = re.search(r'gene_name "([^"]+)"', attr)
            gtype = re.search(r'gene_type "([^"]+)"', attr)

            if not gid or not gname or not gtype:
                continue

            gid = gid.group(1).split(".")[0]
            gname = gname.group(1)
            gtype = gtype.group(1)

            if gtype != "protein_coding":
                continue

            gene_map[gid] = gname
            gene_annot[gname] = (chrom, start, end)

    gene_list = sorted(gene_annot.keys())

    print(f"[INFO] {len(gene_list)} genes loaded")
    return gene_list, gene_map, gene_annot


## GDC download

In [ ]:


def query_files(project, data_type, size=1000):
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "data_type", "value": [data_type]}},
        ],
    }

    params = {
        "filters": json.dumps(filters),
        "format": "JSON",
        "size": str(size),
        "fields": "file_id,file_name,created_datetime",
        "sort": "created_datetime:desc"   
    }

    r = requests.get(GDC_FILES_API, params=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]

    return [(h["file_id"], h["file_name"], h["created_datetime"]) for h in hits]

def download_one(fid, fname, subdir, min_size=1000):
    path = OUT_DIR / subdir / fname
    path.parent.mkdir(parents=True, exist_ok=True)

    # Cache check
    if path.exists():
        size = path.stat().st_size
        if size > min_size:
            print(f"[cache] {fname}")
            return path
        else:
            print(f"[redo] {fname}")
            path.unlink()

    url = f"{GDC_DATA_API}/{fid}"

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

    return path

def parallel_download(files, subdir):
    paths = []
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futures = {ex.submit(download_one, fid, fname, subdir): fname for fid, fname in files}
        for f in as_completed(futures):
            try:
                paths.append(f.result())
            except Exception as e:
                print("[ERROR]", futures[f], e)
    return paths



### GDC API 

Two primary endpoints from the Genomic Data Commons (GDC) are used:

- File query endpoint:
$$
\text{GDC\_API\_FILES} = \texttt{"https://api.gdc.cancer.gov/files"}
$$

- Data download endpoint:
$$
\text{GDC\_API\_DATA} = \texttt{"https://api.gdc.cancer.gov/data"}
$$

In [ ]:
GDC_API_FILES = "https://api.gdc.cancer.gov/files"
GDC_API_DATA = "https://api.gdc.cancer.gov/data"

def get_maf_file_ids(project):
    params = {
        "filters": {
            "op": "and",
            "content": [
                {"op": "in", "content": {"field": "cases.project.project_id", "value": [f"TCGA-{project}"]}},
                {"op": "in", "content": {"field": "data_type", "value": ["Masked Somatic Mutation"]}}
            ]
        },
        "fields": "file_id,file_name",
        "format": "JSON",
        "size": 200
    }

    r = requests.post(GDC_API_FILES, json=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]
    return [(f["file_id"], f["file_name"]) for f in hits]

### Mutation Annotation Format (MAF) files

Let:
- $\mathcal{F} = \{(f_i^{id}, f_i^{name})\}_{i=1}^N$ be the set of file identifiers and names
- $D_p$ be the directory associated with project $p$

This function defines a mapping:

$$
(f_i^{id}, f_i^{name}) \;\longrightarrow\; \text{local file } D_p / f_i^{name}
$$

Each file is retrieved via the GDC data endpoint:

$$
\text{URL}(f_i) = \text{GDC\_API\_DATA} \; / \; f_i^{id}
$$


In [ ]:
def download_maf_files(file_ids, project):
    project_dir = DATA_DIR / project
    project_dir.mkdir(parents=True, exist_ok=True)

    # for file_id, file_name in file_ids:
    for file_id, file_name in tqdm(file_ids, desc=f"Downloading {project}"):
        file_path = project_dir / file_name
        tmp_path = file_path.with_suffix(".tmp")

        if file_path.exists():
            print(f"[SKIP] {file_name}")
            continue

        url = f"{GDC_API_DATA}/{file_id}"
        print(f"[DOWNLOAD] {file_name}")

        try:
            r = requests.get(url, stream=True, timeout=60)

            if r.status_code != 200:
                print(f"[ERROR] Failed {file_id} ({r.status_code})")
                continue

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            tmp_path.rename(file_path)

        except requests.exceptions.RequestException as e:
            print(f"[ERROR] {file_name}: {e}")
            if tmp_path.exists():
                tmp_path.unlink()


### Computation of gene-level mutation frequency across all samples.

Let:
- $S$ be the set of unique samples
- $G$ be the set of genes
- $N = |S|$ be the total number of samples

From all MAF files, we construct the set of unique mutation events:

$$
\mathcal{P} = \{ (g, s) \mid g \text{ is mutated in sample } s \}
$$

Duplicates are removed to ensure each gene–sample pair is counted only once.

The set of samples is:

$$
S = \{ s \mid \exists g \text{ such that } (g,s) \in \mathcal{P} \}
$$

For each gene $g$, the number of mutated samples is:

$$
C_g = \sum_{s \in S} \mathbb{I}((g,s) \in \mathcal{P})
$$

where $\mathbb{I}(\cdot)$ is the indicator function.

The mutation frequency is then computed as:

$$
MF_g = \frac{C_g}{N}
$$

Properties:
- $0 \leq MF_g \leq 1$
- Represents the proportion of samples in which gene $g$ is mutated

In [ ]:
def compute_mutation_frequency(project):
    cancer_dir = DATA_DIR / project

    gene_sample_pairs = set()
    sample_set = set()

    for maf_file in cancer_dir.rglob("*.maf*"):
        try:
            df = pd.read_csv(
                maf_file,
                sep="\t",
                comment="#",
                usecols=["Hugo_Symbol", "Tumor_Sample_Barcode"]
            ).dropna().drop_duplicates()

            pairs = set(zip(df["Hugo_Symbol"], df["Tumor_Sample_Barcode"]))
            gene_sample_pairs.update(pairs)
            sample_set.update(df["Tumor_Sample_Barcode"])

        except:
            continue

    gene_counts = defaultdict(int)
    for gene, sample in gene_sample_pairs:
        gene_counts[gene] += 1

    n = len(sample_set)
    return pd.Series({g: c/n for g, c in gene_counts.items()})

### Load data

In [ ]:

def load_table(path):
    try:
        return pd.read_csv(path, sep="\t", comment="#", low_memory=False)
    except Exception:
        return pd.read_csv(path, low_memory=False)

## Binary Mutation Frequency

Mutation-derived gene-level features are computed from TCGA MAF (Masked Somatic Mutation) files obtained from the Genomic Data Commons (GDC). Mutations are weighted by total mutation burden per sample:

\[
M_i = \log(1 + \text{#mutations in sample } i)
\]

Specifically, binary mutation presence is used to avoid bias like hypermutated samples dominating the signal, and inflating scores for genes simultaneously, which better reflects recurrence across patients.


### Preprocessing steps

1. **Load MAF files**
   - Extract:
     - `Hugo_Symbol` (gene name)
     - sample identifier (e.g., `Tumor_Sample_Barcode`)

2. **Filter mutations**
   - Remove entries with missing values
   - Exclude **silent mutations** to focus on functional variants

3. **Merge all samples**
   - Concatenate all MAF files into a unified dataset



### Sample-level processing

Let:
- \( g \) = gene  
- \( i \) = sample  

We construct a set of unique gene–sample mutation pairs:

\[
(g, i) \in \mathcal{D}
\]

Each pair indicates that gene \( g \) is mutated in sample \( i \), regardless of how many times it appears.


### Gene-Level mutation score

For each gene \( g \), we count the number of samples in which it is mutated:

\[
n_g = |\{ i \mid (g,i) \in \mathcal{D} \}|
\]

Let \( N \) be the total number of samples. The mutation frequency is defined as:

\[
MF_g = \frac{n_g}{N}
\]

In [ ]:
def process_maf(
    paths,
    gene_col="Hugo_Symbol",
    sample_cols=("Tumor_Sample_Barcode", "Sample", "Sample_ID"),
    variant_col="Variant_Classification",
    min_mutations_per_sample=None,
    max_mutations_per_sample=None,  # not needed anymore
    gene_lengths=None,
    log_transform=False,
):

    import pandas as pd
    import numpy as np

    dfs = []
    sample_col_global = None

    # ---------------------------
    # Load and clean MAF files
    # ---------------------------
    for p in paths:
        df = load_table(p)

        if gene_col not in df.columns:
            continue

        sample_col = next((c for c in sample_cols if c in df.columns), None)
        if sample_col is None:
            continue

        sample_col_global = sample_col

        cols = [gene_col, sample_col]
        if variant_col in df.columns:
            cols.append(variant_col)

        df = df[cols].dropna()

        # remove silent mutations
        if variant_col in df.columns:
            df = df[df[variant_col] != "Silent"]

        dfs.append(df)

    if not dfs:
        return pd.Series(dtype=float)

    data = pd.concat(dfs, ignore_index=True)

    # ---------------------------
    # Mutation count per sample (optional filtering)
    # ---------------------------
    sample_counts = data.groupby(sample_col_global)[gene_col].count()

    if min_mutations_per_sample is not None:
        valid = sample_counts[sample_counts >= min_mutations_per_sample].index
        data = data[data[sample_col_global].isin(valid)]

    # recompute after filtering
    sample_counts = data.groupby(sample_col_global)[gene_col].count()
    total_samples = len(sample_counts)

    if total_samples == 0:
        return pd.Series(dtype=float)

    # ---------------------------
    # Binary mutation frequency
    # ---------------------------
    # Step 1: unique (gene, sample) pairs
    gene_sample = (
        data[[gene_col, sample_col_global]]
        .drop_duplicates()
    )

    # Step 2: count number of samples mutated per gene
    gene_scores = (
        gene_sample
        .groupby(gene_col)[sample_col_global]
        .nunique()
    )

    # Step 3: normalize by total samples
    mf = gene_scores / total_samples

    # ---------------------------
    # Gene length normalization
    # ---------------------------
    if gene_lengths is not None:
        gene_len = pd.Series(gene_lengths)
        mf = mf / gene_len
        mf = mf.replace([np.inf, -np.inf], np.nan).dropna()

    # ---------------------------
    # Log transform
    # ---------------------------
    if log_transform:
        mf = np.log1p(mf)

    return mf.sort_values(ascending=False)

### Computation of gene expression scores from RNA-Seq data

Let:
- $E_{g,s}$ be expression of gene $g$ in sample $s$

ID mapping:

$$
\text{Ensembl ID} \rightarrow \text{Gene Symbol}
$$

Log transform:

$$
E'_{g,s} = \log_2(E_{g,s} + 1)
$$

Aggregation:

$$
E_g = \frac{1}{N} \sum_{s=1}^{N} E'_{g,s}
$$

Duplicate genes are averaged:

$$
E_g = \text{mean of duplicated entries}
$$

In [ ]:

def process_expression(paths, gene_map):
    mats = []

    for p in paths:
        df = load_table(p)

        gene_col = next((c for c in df.columns if "gene" in c.lower()), None)
        if gene_col is None:
            continue

        df = df.set_index(gene_col)
        df.index = df.index.astype(str).str.split(".").str[0]
        df.index = df.index.map(gene_map)
        df = df[~df.index.isna()]
        df = df.select_dtypes(include=[np.number])

        mats.append(df)

    if not mats:
        return pd.Series(dtype=float)

    mat = pd.concat(mats, axis=1)
    mat = np.log2(mat + 1)
    mat = mat.groupby(mat.index).mean()

    return mat.mean(axis=1)



## CNA gene-Level scoring

Copy number alteration (CNA) signals are derived from genomic segments and aggregated into gene-level scores using an overlap-weighted strategy, to preserve both the magnitude and direction (amplification vs deletion) of genomic alterations.

### 1. Input representation

Each CNA segment is defined as:

$$
s_i = (\text{chr}_i,\; start_i,\; end_i,\; v_i) \tag{1}
$$

where:
- \( \text{chr}_i \) is the chromosome  
- \( start_i, end_i \) define the genomic interval  
- \( v_i \in \mathbb{R} \) is the CNA value (e.g., log2 copy number ratio)  

Each gene is annotated as:

$$
g = (\text{chr}_g,\; s_g,\; e_g) \tag{2}
$$


### 2. Segment–gene overlap

For a given gene \( g \), we identify all overlapping segments:

$$
\mathcal{S}_g = \{ i \mid start_i \leq e_g \;\wedge\; end_i \geq s_g \} \tag{3}
$$

For each overlapping segment \( i \), the overlap length is:

$$
\ell_i = \max\left(0,\; \min(end_i, e_g) - \max(start_i, s_g)\right) \tag{4}
$$


### 3. Overlap weighting

Each segment is weighted by the fraction of the gene it covers:

$$
w_i = \frac{\ell_i}{e_g - s_g + 1} \tag{5}
$$

This ensures:
- Larger overlaps contribute more  
- Partial overlaps are proportionally weighted  


### 4. Gene-level CNA scores

#### (a) Signed CNA (default)

Preserves biological direction:

$$
\text{CNA}_{\text{signed}}(g) = \sum_{i \in \mathcal{S}_g} w_i \cdot v_i \tag{6}
$$

- \( v_i > 0 \) → amplification  
- \( v_i < 0 \) → deletion  


#### (b) Absolute CNA (magnitude only)

$$
\text{CNA}_{\text{abs}}(g) = \sum_{i \in \mathcal{S}_g} w_i \cdot |v_i| \tag{7}
$$

- Captures intensity of alteration  
- Loses directionality  


#### (c) Amplification frequency

$$
\text{CNA}_{\text{amp}}(g) = \sum_{i \in \mathcal{S}_g} w_i \cdot \mathbb{1}(v_i > 0) \tag{8}
$$

- Measures proportion of gene affected by amplification  


#### (d) Deletion frequency

$$
\text{CNA}_{\text{del}}(g) = \sum_{i \in \mathcal{S}_g} w_i \cdot \mathbb{1}(v_i < 0) \tag{9}
$$

- Measures proportion of gene affected by deletion  


### 5. Output

For each gene \( g \), the method returns one or more scores depending on the selected modes:

- `signed` → biologically meaningful CNA signal (used in models)  
- `abs` → magnitude of alteration  
- `amp` → amplification proportion  
- `del` → deletion proportion  

In [ ]:
def process_cna(paths, gene_annot, return_modes=("signed",)):
    """
    Process CNA segments into gene-level scores.

    Parameters
    ----------
    paths : list
        List of CNA segment file paths
    gene_annot : dict
        {gene: (chr, start, end)}
    return_modes : tuple
        Options: "signed", "abs", "amp", "del"

    Returns
    -------
    dict of pd.Series
    """
    import numpy as np
    import pandas as pd

    segs = []

    # -----------------------------
    # Load + standardize segments
    # -----------------------------
    for p in paths:
        df = load_table(p)

        if not all(c in df.columns for c in ["Chromosome", "Start", "End"]):
            continue

        val_col = next(
            (c for c in df.columns if "mean" in c.lower() or "value" in c.lower()),
            None,
        )
        if val_col is None:
            continue

        tmp = df[["Chromosome", "Start", "End", val_col]].copy()
        tmp.columns = ["chr", "start", "end", "val"]
        tmp["chr"] = tmp["chr"].astype(str).str.replace("chr", "")

        segs.append(tmp)

    if not segs:
        return {mode: pd.Series(dtype=float) for mode in return_modes}

    seg = pd.concat(segs, ignore_index=True)

    # -----------------------------
    # Pre-sort for efficiency
    # -----------------------------
    seg = seg.sort_values(["chr", "start"])

    gene_vals = {mode: {} for mode in return_modes}

    # -----------------------------
    # Main loop (gene-wise)
    # -----------------------------
    for g, (chr_, s, e) in gene_annot.items():
        chr_ = str(chr_).replace("chr", "")

        seg_chr = seg[seg["chr"] == chr_]
        if seg_chr.empty:
            for mode in return_modes:
                gene_vals[mode][g] = 0.0
            continue

        ov = seg_chr[
            (seg_chr["start"] <= e) &
            (seg_chr["end"] >= s)
        ]

        if ov.empty:
            for mode in return_modes:
                gene_vals[mode][g] = 0.0
            continue

        # -----------------------------
        # Compute overlap length
        # -----------------------------
        overlap_len = np.minimum(ov["end"], e) - np.maximum(ov["start"], s)
        overlap_len = np.maximum(overlap_len, 0)

        weights = overlap_len / (e - s + 1)

        vals = ov["val"].values

        # -----------------------------
        # Compute outputs
        # -----------------------------
        if "signed" in return_modes:
            gene_vals["signed"][g] = np.sum(weights * vals)

        if "abs" in return_modes:
            gene_vals["abs"][g] = np.sum(weights * np.abs(vals))

        if "amp" in return_modes:
            gene_vals["amp"][g] = np.sum(weights * (vals > 0))

        if "del" in return_modes:
            gene_vals["del"][g] = np.sum(weights * (vals < 0))

    # -----------------------------
    # Convert to Series
    # -----------------------------
    return {
        mode: pd.Series(gene_vals[mode])
        for mode in return_modes
    }

### Computatiion of miRNA expression levels

Let:
- $R_{r,s}$ be expression of miRNA $r$ in sample $s$

Aggregation:

$$
R_r = \frac{1}{N} \sum_{s=1}^{N} R_{r,s}
$$

Log transform:

$$
R'_r = \log_2(R_r + 1)
$$


In [ ]:

def process_mirna(paths):
    mats = []

    for p in paths:
        df = load_table(p)

        mirna_col = next((c for c in df.columns if "mirna" in c.lower()), None)
        val_col = next((c for c in df.columns if "read" in c.lower() or "rpm" in c.lower()), None)

        if mirna_col is None or val_col is None:
            continue

        tmp = df[[mirna_col, val_col]].copy()
        tmp.columns = ["mirna", "value"]
        tmp = tmp.dropna()

        mats.append(tmp)

    if not mats:
        return pd.Series(dtype=float)

    mat = pd.concat(mats)
    mirna_expr = mat.groupby("mirna")["value"].mean()

    return np.log2(mirna_expr + 1)




### Load miRNA–gene interactions

Let:
- $\mathcal{T}(r)$ be the set of target genes for miRNA $r$
- $w_{r,g}$ be interaction weight

Mapping:

$$
\mathcal{T}(r) = \{ (g, w_{r,g}) \}
$$

Multiple sources can be combined:

$$
\mathcal{T}(r) = \bigcup_k \mathcal{T}_k(r)
$$


In [ ]:


# =========================================================
# miRNA TARGETS
# =========================================================
def load_mirtarbase(mirtarbase_files):
    mapping = {}

    if isinstance(mirtarbase_files, str):
        mirtarbase_files = {"default": (mirtarbase_files, 1.0)}

    for _, (path, weight) in mirtarbase_files.items():

        df = pd.read_excel(path) if path.endswith(".xlsx") else pd.read_csv(path)

        mir_col = next(c for c in df.columns if "miRNA" in c)
        gene_col = next(c for c in df.columns if "Target Gene" in c)

        for _, row in df.iterrows():
            mir = str(row[mir_col]).lower().strip()
            gene = str(row[gene_col]).upper().strip()

            mapping.setdefault(mir, []).append((gene, weight))

    return mapping


### Map miRNA expression to gene-level scores

Let:
- $R_r$ be miRNA expression
- $w_{r,g}$ be interaction weight

Weighted sum:

$$
S_g = \sum_{r \in \mathcal{R}_g} w_{r,g} \cdot R_r
$$

Normalization:

$$
R_g = \frac{S_g}{\sum_{r \in \mathcal{R}_g} w_{r,g}}
$$

If no interactions:

$$
R_g = 0
$$


In [ ]:
def map_mirna_to_genes_weighted(mirna_series, gene_list, mirna_targets):
    gene_scores = {g: 0.0 for g in gene_list}
    gene_weights = {g: 0.0 for g in gene_list}

    for mir, val in mirna_series.items():
        mir = mir.lower().strip()

        if mir not in mirna_targets:
            continue

        for gene, weight in mirna_targets[mir]:
            if gene in gene_scores:
                gene_scores[gene] += val * weight
                gene_weights[gene] += weight

    return pd.Series({
        g: gene_scores[g] / gene_weights[g] if gene_weights[g] > 0 else 0.0
        for g in gene_list
    })



### Min-max normalization

Let $x_g$ denote the value for gene $g$.

Missing values are first imputed:

$$
x_g =
\begin{cases}
x_g, & \text{if observed} \\
0, & \text{if missing}
\end{cases}
$$

Normalization:

$$
\tilde{x}_g = \frac{x_g - \min(x)}{\max(x) - \min(x) + \epsilon}
$$

where $\epsilon = 10^{-12}$ ensures numerical stability.

In [ ]:

def minmax(s):
    s = s.fillna(0)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


### Retrieve the latest GENCODE annotation version

Let:
- $\mathcal{V} = \{v_1, v_2, \dots, v_k\}$ be the set of available versions

The latest version is selected as:

$$
v^* = \max(\mathcal{V})
$$

The corresponding annotation file URL is constructed as:

$$
\text{URL} = \text{base} \; / \; \text{release}_{v^*} \; / \; \text{gencode.v}_{v^*}.annotation.gtf.gz
$$

This ensures:
- Use of the most recent gene annotation
- Consistency with current genomic references

Thus, the function implements:

$$
\mathcal{V} \rightarrow v^* \rightarrow \text{annotation file}
$$

In [ ]:

def get_latest_gencode_url():
    base = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/"

    r = requests.get(base)
    r.raise_for_status()

    versions = re.findall(r"release_(\d+)", r.text)
    latest = max(map(int, versions))

    url = f"{base}/release_{latest}/gencode.v{latest}.annotation.gtf.gz"

    print(f"[INFO] Latest GENCODE: v{latest}")
    return url, f"v{latest}"


### Construction of multi-omics features

Let:
- $G$ be the set of genes
- $\mathcal{M} = \{\text{mutation}, \text{expression}, \text{CNA}, \text{miRNA}\}$

For each modality $m \in \mathcal{M}$, raw data are processed:

$$
\begin{aligned}
MF_g &= \text{process\_maf}(\cdot) \\
E_g  &= \text{process\_expression}(\cdot) \\
C_g  &= \text{process\_cna}(\cdot) \\
R_g  &= \text{process\_mirna}(\cdot)
\end{aligned}
$$

miRNA is mapped to genes:

$$
R_g = \text{map\_mirna\_to\_genes}(R_r)
$$

All features are aligned to a common gene set:

$$
x_g^{(m)} = \text{reindex}(G)
$$

and normalized:

$$
\tilde{x}_g^{(m)} = \text{minmax}(x_g^{(m)})
$$

A validity mask is applied to retain informative genes:

$$
\mathcal{G}' = \{ g \in G \mid MF_g \neq 0 \lor E_g \neq 0 \lor C_g \neq 0 \lor R_g \neq 0 \}
$$

Final output consists of modality-specific matrices:

$$
X^{(m)} = \{ \tilde{x}_g^{(m)} \mid g \in \mathcal{G}' \}
$$

Thus, the function implements:

$$
\text{Raw Multi-Omics Data} \rightarrow \text{Processed Feature Matrices}
$$

for each cancer type.

In [ ]:
def process_cancer(project, gene_list, gene_map, gene_annot, gene_lengths):

    print(f"\n=== {project} ===")

    paths = {}
    for k, v in DATA_TYPES.items():

        files = query_files(project, v)
        files = files[:FILE_LIMITS[k]]

        paths[k] = parallel_download(
            [(fid, fname) for fid, fname, _ in files],
            f"{project}/{k}"
        )

    # -----------------------------
    # External resources
    # -----------------------------
    mirna_targets = load_mirtarbase(MIRTARBASE_FILES)
    short = project.replace("TCGA-", "")

    # -----------------------------
    # Compute modalities
    # -----------------------------
    mut = process_maf(
        paths["mutation"],
        gene_lengths=gene_lengths,
        max_mutations_per_sample=1000,
        log_transform=True
    )

    expr = process_expression(paths["expression"], gene_map)

    # 🔥 NEW CNA (multi-output)
    cna_dict = process_cna(
        paths["cna"],
        gene_annot,
        return_modes=("signed", "amp", "del")
    )

    cna_signed = cna_dict["signed"]
    cna_amp = cna_dict["amp"]
    cna_del = cna_dict["del"]

    # miRNA
    mirna_raw = process_mirna(paths["mirna"])
    mirna = map_mirna_to_genes_weighted(mirna_raw, gene_list, mirna_targets)

    # -----------------------------
    # Align to gene list
    # -----------------------------
    mut = mut.reindex(gene_list).fillna(0)
    expr = expr.reindex(gene_list).fillna(0)
    cna_signed = cna_signed.reindex(gene_list).fillna(0)
    cna_amp = cna_amp.reindex(gene_list).fillna(0)
    cna_del = cna_del.reindex(gene_list).fillna(0)
    mirna = mirna.reindex(gene_list).fillna(0)

    # -----------------------------
    # Normalize
    # -----------------------------
    mut = minmax(mut)
    expr = minmax(expr)

    # ⚠️ IMPORTANT: handle CNA differently
    # signed → keep direction but scale
    cna_signed = cna_signed / (np.max(np.abs(cna_signed)) + 1e-8)

    # amp/del already [0,1]-like → optional minmax
    cna_amp = minmax(cna_amp)
    cna_del = minmax(cna_del)

    mirna = minmax(mirna)

    # -----------------------------
    # Mask (keep informative genes)
    # -----------------------------
    mask = (
        (mut != 0) |
        (expr != 0) |
        (cna_signed != 0) |
        (mirna != 0)
    )

    # -----------------------------
    # Return
    # -----------------------------
    return {
        # Core features (for GNN)
        "MF": pd.DataFrame({short: mut[mask]}),
        "GE": pd.DataFrame({short: expr[mask]}),
        "CNA": pd.DataFrame({short: cna_signed[mask]}),
        "miRNA": pd.DataFrame({short: mirna[mask]}),

        # 🔥 Extra outputs (for visualization & interpretation)
        "CNA_amp": pd.DataFrame({short: cna_amp[mask]}),
        "CNA_del": pd.DataFrame({short: cna_del[mask]}),
    }

### Construction of multi-omics feature matrices

The function implements the transformation:

$$
\text{Raw Multi-Omics Data}
\;\longrightarrow\;
\text{Aligned and Normalized Feature Matrices}
$$

with the following properties:
- Multi-modal integration across mutation, expression, CNA, and miRNA
- Consistent gene indexing across all modalities
- Independent normalization per modality
- Removal of non-informative (all-zero) genes
- Deterministic and reproducible processing

Let:
- $G$ be the set of genes
- $\mathcal{M} = \{\text{mutation}, \text{expression}, \text{CNA}, \text{miRNA}\}$ be the set of modalities


### 1: Modality-specific feature computation

For each modality $m \in \mathcal{M}$, gene-level scores are computed:

$$
\begin{aligned}
MF_g &= \text{process\_maf}(\cdot) \\
E_g  &= \text{process\_expression}(\cdot) \\
C_g  &= \text{process\_cna}(\cdot) \\
R_r  &= \text{process\_mirna}(\cdot)
\end{aligned}
$$

miRNA values are defined at the miRNA level $r$, and are mapped to gene-level scores:

$$
R_g = \text{map\_mirna\_to\_genes}(R_r)
$$

### 2: Alignment to a common gene space

All modality-specific vectors are aligned to the same gene index:

$$
x_g^{(m)} =
\begin{cases}
\text{observed value}, & g \in \text{data} \\
0, & \text{otherwise}
\end{cases}
\quad \forall g \in G
$$

This ensures consistent dimensionality across modalities.


### 3: Normalization

Each modality is independently normalized using min-max scaling:

$$
\tilde{x}_g^{(m)} = \frac{x_g^{(m)} - \min(x^{(m)})}{\max(x^{(m)}) - \min(x^{(m)}) + \epsilon}
$$

where $\epsilon$ is a small constant for numerical stability.


### 4: Informative gene filtering

A binary mask is applied to retain genes with at least one non-zero signal:

$$
\mathcal{G}' = \left\{ g \in G \;\middle|\; \exists m \in \mathcal{M} \text{ such that } \tilde{x}_g^{(m)} > 0 \right\}
$$

This removes genes with no observed signal across all modalities.


### 5: Output construction

For each modality $m$, the final feature matrix is:

$$
X^{(m)} = \{ \tilde{x}_g^{(m)} \mid g \in \mathcal{G}' \}
$$

Each $X^{(m)}$ is a column vector indexed by genes and associated with the given cancer type.


### Final representation

For each gene $g \in \mathcal{G}'$, the multi-omics feature vector is:

$$
\mathbf{x}_g = \left[ \tilde{MF}_g, \tilde{E}_g, \tilde{C}_g, \tilde{R}_g \right]
$$

In [ ]:
from tqdm.auto import tqdm
import pandas as pd


gene_list, gene_map, gene_annot = parse_gtf()

print("\nGDC file versions...")

gdc_versions = {
    c: {
        k: [
            {
                "file_id": f[0],
                "file_name": f[1],
                "created_datetime": f[2]
            }
            for f in query_files(c, DATA_TYPES[k])[:3]
        ]
        for k in DATA_TYPES
    }
    for c in CANCERS
}

print("[INFO] GDC versions captured")

gene_lengths = {g: e - s for g, (_, s, e) in gene_annot.items()}

MF_all = pd.DataFrame(index=gene_list)
GE_all = pd.DataFrame(index=gene_list)
CNA_all = pd.DataFrame(index=gene_list)
mirna_all = pd.DataFrame(index=gene_list)

for c in tqdm(CANCERS, desc="Processing cancers"):
    res = process_cancer(c, gene_list, gene_map, gene_annot, gene_lengths)

    MF_all = pd.concat([MF_all, res["MF"]], axis=1)
    GE_all = pd.concat([GE_all, res["GE"]], axis=1)
    CNA_all = pd.concat([CNA_all, res["CNA"]], axis=1)
    mirna_all = pd.concat([mirna_all, res["miRNA"]], axis=1)

# filter
mask = (
    (MF_all.sum(axis=1) > 0) &
    (GE_all.sum(axis=1) > 0) &
    (CNA_all.sum(axis=1) > 0) &
    (mirna_all.sum(axis=1) > 0)
)

MF_all = MF_all.loc[mask]
GE_all = GE_all.loc[mask]
CNA_all = CNA_all.loc[mask]
mirna_all = mirna_all.loc[mask]

print(f"[FINAL] Kept {mask.sum()} genes")

# save
MF_all.to_csv(OUTPUT_DIR / "mutation_features.csv")
GE_all.to_csv(OUTPUT_DIR / "expression_features.csv")
CNA_all.to_csv(OUTPUT_DIR / "cna_features.csv")
mirna_all.to_csv(OUTPUT_DIR / "mirna_features.csv")

print("Saved all omics features.")


### GDC metadata snapshot (Data version tracking)

This step captures a snapshot of GDC file metadata to ensure reproducibility.

Let:
- $\mathcal{C}$ be the set of cancer types
- $\mathcal{M}$ be the set of data modalities
- $\mathcal{F}_{c,m}$ be the set of files returned by the GDC API for cancer $c$ and modality $m$

From each set $\mathcal{F}_{c,m}$, we select the top-$k$ most recent files (here $k=3$):

$$
\mathcal{F}_{c,m}^{(k)} = \{ f_1, f_2, \dots, f_k \}
$$

where files are ordered by creation timestamp:

$$
t(f_1) \geq t(f_2) \geq \dots \geq t(f_k)
$$

For each file $f$, we extract metadata:

$$
f \mapsto \{ \text{file\_id}, \text{file\_name}, \text{created\_datetime} \}
$$

The resulting structure is:

$$
\mathcal{V} = \left\{ (c, m) \rightarrow \mathcal{F}_{c,m}^{(k)} \right\}
$$

This metadata snapshot provides:

- Version tracking of all input data
- Traceability of file provenance
- Reproducibility of dataset selection

Thus, the process implements:

$$
\text{GDC Query} \;\longrightarrow\; \text{Metadata Snapshot} \;\longrightarrow\; \text{Reproducible Analysis}
$$

In [ ]:


print("\n GDC file versions (metadata snapshot)...")

gdc_file_versions = {
    c: {
        k: [
            {
                "file_id": f[0],
                "file_name": f[1],
                "created_datetime": f[2]
            }
            for f in query_files(c, DATA_TYPES[k])[:3]
        ]
        for k in DATA_TYPES
    }
    for c in CANCERS
}

print("[INFO] GDC version snapshot complete")

### GDC metadata collection and manifest generation

Construct a complete metadata manifest to ensure reproducibility of the dataset and analysis:
- Full reproducibility of data selection
- Traceability of all source files
- Documentation of pipeline configuration


### 1. File metadata aggregation

Let:
- $\mathcal{C}$ be the set of cancer types
- $\mathcal{M}$ be the set of data modalities
- $\mathcal{F}_{c,m}$ be the set of files returned by the GDC API for cancer $c$ and modality $m$

Each file $f \in \mathcal{F}_{c,m}$ is associated with:
- $\text{file\_id}(f)$
- $\text{file\_name}(f)$
- $\text{created\_datetime}(f)$

Files are sorted by timestamp:

$$
\mathcal{F}_{c,m}^{\text{sorted}} = \text{sort}(\mathcal{F}_{c,m}, \ \text{by } t(f) \text{ descending})
$$

We extract the top-$k$ most recent files (here $k=3$):

$$
\mathcal{F}_{c,m}^{(k)} = \{ f_1, f_2, \dots, f_k \}
$$

and record the total number of available files:

$$
N_{c,m} = |\mathcal{F}_{c,m}|
$$

Thus, the metadata structure for each $(c,m)$ is:

$$
\mathcal{V}_{c,m} =
\left\{
\text{top\_k\_latest}: \mathcal{F}_{c,m}^{(k)},
\quad
\text{total\_available}: N_{c,m}
\right\}
$$


### 2. Global metadata structure

The full metadata collection is:

$$
\mathcal{V} = \{ (c, m) \rightarrow \mathcal{V}_{c,m} \mid c \in \mathcal{C}, m \in \mathcal{M} \}
$$

This provides:
- Complete visibility into available data
- Explicit tracking of selected files
- Robust handling of missing or failed queries


### 3. Manifest definition

A global manifest $\mathcal{M}$ is constructed to capture all relevant information:

$$
\mathcal{M} =
\{
\text{date},
\text{GENCODE version},
\text{GDC query policy},
\text{dataset dimensions},
\text{directory structure},
\mathcal{V}
\}
$$

Specifically:
- Timestamp:
$$
t = \text{datetime.now()}
$$

- Number of genes:
$$
|G'| = \sum_{g \in G} \mathbb{I}(g \text{ retained after filtering})
$$

- Number of cancer types:
$$
|\mathcal{C}|
$$


### 4. Reproducibility guarantee

The manifest is serialized to disk:

$$
\mathcal{M} \rightarrow \text{JSON file}
$$

In [ ]:
from datetime import datetime

print("[INFO] GDC file version metadata...")

gdc_file_versions = {}

for c in tqdm(CANCERS, desc="Collecting GDC metadata"):
    gdc_file_versions[c] = {}

    for k in DATA_TYPES:
        try:
            files = query_files(c, DATA_TYPES[k])

            gdc_file_versions[c][k] = {
                "top_3_latest": [
                    {
                        "file_id": f[0],
                        "file_name": f[1],
                        "created_datetime": f[2]
                    }
                    for f in files[:3]
                ],
                "total_available": len(files)
            }

        except Exception as e:
            gdc_file_versions[c][k] = {"error": str(e)}

manifest = {
    "date": str(datetime.now()),
    "gencode_version": GENCODE_VERSION,
    "gdc_query_sorted_by": "created_datetime DESC",
    "num_genes_final": int(mask.sum()),
    "num_cancers": len(CANCERS),
    "cancers": CANCERS,
    "data_types": list(DATA_TYPES.keys()),
    "file_limits": FILE_LIMITS,

    "directories": {
        "OUT_DIR": str(OUT_DIR),
        "DATA_DIR": str(DATA_DIR),
        "OUTPUT_DIR": str(OUTPUT_DIR),
    },

    "gdc_file_versions": gdc_file_versions,

    "notes": (
        "Fully reproducible pipeline using latest GENCODE and "
        "GDC files sorted by created_datetime DESC"
    )
}

with open(OUTPUT_DIR / "data_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("[DONE] Manifest saved.")

### Uniform Manifold Approximation and Projection (UMAP) for gene features

We use Uniform Manifold Approximation and Projection (UMAP) to visualize high-dimensional gene features in a low-dimensional space.

Let:
- $\mathbf{x}_g \in \mathbb{R}^d$ be the feature vector for gene $g$
- $X \in \mathbb{R}^{n \times d}$ be the full feature matrix

UMAP constructs a graph representation in high-dimensional space by estimating pairwise similarities:

$$
p_{ij} = \exp\left(-\frac{\| \mathbf{x}_i - \mathbf{x}_j \|^2}{\sigma_i}\right)
$$

where $\sigma_i$ controls local neighborhood size.

It then learns a low-dimensional embedding $\mathbf{y}_i \in \mathbb{R}^2$ by minimizing the cross-entropy between high- and low-dimensional graphs:

$$
\mathcal{L} = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

where $q_{ij}$ is the similarity in embedding space.

The result is a mapping:

$$
X \in \mathbb{R}^{n \times d} \;\longrightarrow\; Y \in \mathbb{R}^{n \times 2}
$$

which preserves both local and global structure of the data.

In [ ]:
COLORS = {
    0: '#0077B6',   1: '#0000FF',   2: '#00B4D8',   3: '#48EAC4',
    4: '#F1C0E8',   5: '#B9FBC0',   6: '#32CD32',   7: '#BEE1E6',
    8: '#8A2BE2',   9: '#E377C2',  10: '#8EECF5',  11: '#A3C4F3',
    12: '#FFB347', 13: '#FFD700',  14: '#FF69B4',  15: '#CD5C5C',
    16: '#7FFFD4', 17: '#FF7F50',  18: '#C71585',  19: '#20B2AA',
    20: '#6A5ACD', 21: '#40E0D0',  22: '#FF8C00',  23: '#DC143C',
    24: '#9ACD32',
    25: '#1F77B4', 26: '#FF1493', 27: '#2E8B57', 28: '#D2691E',
    29: '#9932CC', 30: '#00CED1', 31: '#FF4500', 32: '#708090'
}


disease_to_tcga = {
    "adrenocortical carcinoma": "ACC",
    "bladder urothelial carcinoma": "BLCA",
    "breast invasive carcinoma": "BRCA",
    "cervical squamous cell carcinoma and endocervical adenocarcinoma": "CESC",
    "cholangiocarcinoma": "CHOL",
    "colon adenocarcinoma": "COAD",
    "rectum adenocarcinoma": "READ",
    "diffuse large B-cell lymphoma": "DLBC",
    "esophageal carcinoma": "ESCA",
    "glioblastoma multiforme": "GBM",
    "head and neck squamous cell carcinoma": "HNSC",
    "kidney chromophobe": "KICH",
    "kidney renal clear cell carcinoma": "KIRC",
    "kidney renal papillary cell carcinoma": "KIRP",
    "acute myeloid leukemia": "LAML",
    "brain lower grade glioma": "LGG",
    "liver hepatocellular carcinoma": "LIHC",
    "lung adenocarcinoma": "LUAD",
    "lung squamous cell carcinoma": "LUSC",
    "mesothelioma": "MESO",
    "ovarian serous cystadenocarcinoma": "OV",
    "pancreatic adenocarcinoma": "PAAD",
    "pheochromocytoma and paraganglioma": "PCPG",
    "prostate adenocarcinoma": "PRAD",
    "sarcoma": "SARC",
    "skin cutaneous melanoma": "SKCM",
    "stomach adenocarcinoma": "STAD",
    "testicular germ cell tumors": "TGCT",
    "thyroid carcinoma": "THCA",
    "thymoma": "THYM",
    "uterine corpus endometrial carcinoma": "UCEC",
    "uterine carcinosarcoma": "UCS",
    "uveal melanoma": "UVM"
}

len(set(disease_to_tcga.values()))

## Gene-Level Visualization

Transform high-dimensional multi-omics gene features into a 2D embedding that:

- Preserves local biological similarity  
- Enables cluster visualization  
- Highlights dominant cancer-specific signals  

### Feature representation

Each gene $g$ is represented by a high-dimensional feature vector:

$$
\mathbf{x}_g \in \mathbb{R}^d
\tag{1}
$$

where the feature vector integrates multi-omics signals across cancers:

$$
\mathbf{x}_g = [\text{MF}, \text{GE}, \text{CNA}, \text{miRNA}]_{\text{across cancers}}
\tag{2}
$$

### Preprocessing

To ensure comparability across features, standardization is applied:

$$
\tilde{\mathbf{x}}_g = \frac{\mathbf{x}_g - \mu}{\sigma}
\tag{3}
$$

where:

$$
\mu = \text{mean across samples}
\tag{4}
$$

$$
\sigma = \text{standard deviation across samples}
\tag{5}
$$


### Dimensionality reduction (PCA)

Principal Component Analysis (PCA) projects the standardized features into a lower-dimensional subspace:

$$
\mathbf{z}_g = \mathbf{W}^\top \tilde{\mathbf{x}}_g
\tag{6}
$$

where:

$$
\mathbf{W} \in \mathbb{R}^{d \times k}
\tag{7}
$$

$$
\mathbf{z}_g \in \mathbb{R}^k
\tag{8}
$$


### UMAP embedding

UMAP constructs a low-dimensional embedding:

$$
\mathbf{y}_g \in \mathbb{R}^2
\tag{9}
$$

UMAP models pairwise similarities in high-dimensional space as:

$$
p_{ij} = \exp\left(-\frac{\| \mathbf{x}_i - \mathbf{x}_j \|^2}{\sigma_i}\right)
\tag{10}
$$

and similarities in the embedding space as:

$$
q_{ij} = \frac{1}{1 + a \| \mathbf{y}_i - \mathbf{y}_j \|^{2b}}
\tag{11}
$$

The embedding is obtained by minimizing the cross-entropy:

$$
\mathcal{L} =
\sum_{i \neq j}
\left[
p_{ij} \log \frac{p_{ij}}{q_{ij}} +
(1 - p_{ij}) \log \frac{1 - p_{ij}}{1 - q_{ij}}
\right]
\tag{12}
$$


Each point in the scatter plot corresponds to a gene:

**Position:**

$$
(UMAP1, UMAP2) = \mathbf{y}_g
\tag{13}
$$


### Jitter for stability

To reduce overplotting, a small Gaussian perturbation is added:

$$
\mathbf{y}_g' = \mathbf{y}_g + \boldsymbol{\epsilon}
\tag{14}
$$

$$
\boldsymbol{\epsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I})
\tag{15}
$$

In [ ]:
# Combine features
X = pd.concat(
    [
        MF_all.add_prefix("MF_"),
        GE_all.add_prefix("GE_"),
        CNA_all.add_prefix("CNA_"),
        mirna_all.add_prefix("miRNA_"),
    ],
    axis=1
)

# Clean
X = X.replace([np.inf, -np.inf], np.nan).dropna()

print("X shape:", X.shape)

In [ ]:
# Extract cancer names from columns
cancer_labels = np.array([c.split("_")[-1] for c in X.columns])

# Assign dominant cancer per gene
dominant_idx = np.argmax(X.values, axis=1)
gene_cancer = cancer_labels[dominant_idx]

labels = pd.Series(gene_cancer, index=X.index)

In [ ]:
mut_signal = X[[c for c in X.columns if c.startswith("MF_")]].sum(axis=1)
expr_signal = X[[c for c in X.columns if c.startswith("GE_")]].sum(axis=1)

gene_type = np.where(mut_signal > expr_signal, "Mutation-driven", "Expression-driven")
gene_type = pd.Series(gene_type, index=X.index)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.decomposition import PCA

pca = PCA(n_components=20, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance (first 50 PCs):",
      np.sum(pca.explained_variance_ratio_))

import umap

umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

umap_embedding = umap_model.fit_transform(X_pca)


In [ ]:
plot_df = pd.DataFrame(
    umap_embedding,
    columns=["UMAP1", "UMAP2"],
    index=X.index   # genes
)

# add labels
plot_df["Cancer"] = labels.loc[plot_df.index]
plot_df["Type"] = gene_type.loc[plot_df.index]

print("Missing labels:", plot_df["Cancer"].isna().sum())
print("Cancer types:", plot_df["Cancer"].nunique())
print(plot_df["Type"].value_counts())



In [ ]:
plot_df["UMAP1_j"] = plot_df["UMAP1"] + np.random.normal(0, 0.1, len(plot_df))
plot_df["UMAP2_j"] = plot_df["UMAP2"] + np.random.normal(0, 0.1, len(plot_df))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10))

sns.scatterplot(
    data=plot_df,
    x="UMAP1_j",
    y="UMAP2_j",

    hue="Cancer",
    style="Type",

    palette=sns.color_palette(
        "tab20",
        n_colors=plot_df["Cancer"].nunique()
    ),

    s=20,                # smaller for genes
    alpha=0.7,

    edgecolor="black",
    linewidth=0.2
)

plt.xlabel("UMAP-1", fontsize=14)
plt.ylabel("UMAP-2", fontsize=14)
plt.title("Gene-Level UMAP (Multi-Omics Integration)", fontsize=18)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=12,
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# ======================================================
# Plot 1: Colored by Cancer
# ======================================================

plt.figure(figsize=(14, 10))

sns.scatterplot(
    data=plot_df,
    x="UMAP1_j",
    y="UMAP2_j",

    hue="Cancer",

    palette=sns.color_palette(
        "tab20",
        n_colors=plot_df["Cancer"].nunique()
    ),

    s=20,
    alpha=0.7,

    edgecolor="black",
    linewidth=0.2
)

plt.xlabel("UMAP-1", fontsize=14)
plt.ylabel("UMAP-2", fontsize=14)

plt.title(
    "Gene-Level UMAP Colored by Cancer",
    fontsize=18
)

# =========================
# Legend INSIDE upper right
# =========================
plt.legend(
    loc="upper right",
    fontsize=10,
    frameon=True
)

plt.tight_layout()

# ------------------------------------------------------
# Save cancer plot
# ------------------------------------------------------
plt.savefig(
    RESULT_DIR / "umap_by_cancer.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: umap_by_cancer.png")

# ======================================================
# Plot 2: Colored by Omics Type
# ======================================================

plt.figure(figsize=(14, 10))

sns.scatterplot(
    data=plot_df,
    x="UMAP1_j",
    y="UMAP2_j",

    hue="Type",
    style="Type",

    palette="Set2",

    s=20,
    alpha=0.7,

    edgecolor="black",
    linewidth=0.2
)

plt.xlabel("UMAP-1", fontsize=14)
plt.ylabel("UMAP-2", fontsize=14)

plt.title(
    "Gene-Level UMAP Colored by Omics Type",
    fontsize=18
)

# =========================
# Legend INSIDE upper right
# =========================
plt.legend(
    loc="upper right",
    fontsize=12,
    frameon=True
)

plt.tight_layout()

# ------------------------------------------------------
# Save omics plot
# ------------------------------------------------------
plt.savefig(
    RESULT_DIR / "umap_by_omics_type.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: umap_by_omics_type.png")

## Gene-Level Faceted UMAP Visualization

This figure presents a **faceted UMAP visualization**, where each panel corresponds to a different omics layer (MF, GE, CNA, miRNA), while sharing the same gene space.

---

### Feature Representation

Each gene \( g \) is represented by a high-dimensional feature vector:

$$
\mathbf{x}_g \in \mathbb{R}^d \tag{1}
$$

where

$$
\mathbf{x}_g = [\text{MF}, \text{GE}, \text{CNA}, \text{miRNA}]_{\text{across cancers}} \tag{2}
$$

---

### Preprocessing

To ensure comparability across features, standardization is applied:

$$
\tilde{\mathbf{x}}_g = \frac{\mathbf{x}_g - \mu}{\sigma} \tag{3}
$$

Dimensionality reduction via PCA:

$$
\mathbf{z}_g = \mathbf{W}^\top \tilde{\mathbf{x}}_g \tag{4}
$$

---

### Shared UMAP Embedding

A single UMAP embedding is learned:

$$
\mathbf{y}_g \in \mathbb{R}^2 \tag{5}
$$

UMAP models pairwise similarities in the original space as:

$$
p_{ij} = \exp\left(-\frac{\| \mathbf{x}_i - \mathbf{x}_j \|^2}{\sigma_i}\right) \tag{6}
$$

and in the embedding space as:

$$
q_{ij} = \frac{1}{1 + a \| \mathbf{y}_i - \mathbf{y}_j \|^{2b}} \tag{7}
$$

The embedding is obtained by minimizing:

$$
\mathcal{L} = \sum_{i \neq j}
\left[
p_{ij} \log \frac{p_{ij}}{q_{ij}} +
(1 - p_{ij}) \log \frac{1 - p_{ij}}{1 - q_{ij}}
\right] \tag{8}
$$

---

### Faceted Visualization

Instead of plotting all signals together, the embedding is **faceted across omics types**:

$$
\text{Panels} = \{\text{MF}, \text{GE}, \text{CNA}, \text{miRNA}\} \tag{9}
$$

Each panel:
- Uses the same coordinates \( \mathbf{y}_g \)
- Colors genes by the corresponding omics signal

---

### Visualization Mapping

For each panel:

- **Position**:

$$
(UMAP1, UMAP2) = \mathbf{y}_g \tag{10}
$$

- **Color (hue)**: omics-specific signal intensity  
- **Shared layout across panels** enables direct comparison  

---


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# Ensure clean dataframe
# =========================
df = plot_df.copy()

df = df.replace([np.inf, -np.inf], np.nan).dropna()

# =========================
# FacetGrid (OMICS PANELS)
# =========================
g = sns.FacetGrid(
    df,
    col="Type",          # MF / GE / CNA / miRNA
    col_wrap=2,          # 2x2 layout
    height=5,
    sharex=True,
    sharey=True
)

# =========================
# Scatter plot per facet
# =========================
g.map_dataframe(
    sns.scatterplot,
    x="UMAP1_j",
    y="UMAP2_j",
    hue="Cancer",
    palette=sns.color_palette(
        "tab20",
        df["Cancer"].nunique()
    ),
    s=15,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.2
)

# =========================
# Styling
# =========================
g.set_axis_labels("UMAP-1", "UMAP-2")
g.set_titles(col_template="{col_name}")

# =========================
# Remove duplicate legends
# =========================
for ax in g.axes.flat:

    legend = ax.get_legend()

    if legend is not None:
        legend.remove()

# =========================
# Single INSIDE legend
# =========================
handles, labels = g.axes[0].get_legend_handles_labels()

g.fig.legend(
    handles,
    labels,
    loc="upper right",
    fontsize=9,
    frameon=True,
    ncol=1
)

plt.subplots_adjust(top=0.92)

# g.fig.suptitle(
#     "Faceted Gene UMAP Across Multi-Omics Layers",
#     fontsize=16
# )

plt.tight_layout()

# ------------------------------------------------------
# Save figure
# ------------------------------------------------------
save_path = RESULT_DIR / "faceted_multiomics_umap.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"✅ Saved figure to: {save_path}")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Ensure clean dataframe
# =========================
df = plot_df.copy()

df = df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

# =========================
# FacetGrid (VERTICAL layout)
# =========================
g = sns.FacetGrid(
    df,
    col="Type",
    col_wrap=1,          # vertical stacking
    height=5,
    aspect=1.2,
    sharex=True,
    sharey=True
)

# =========================
# Scatter plot per facet
# =========================
g.map_dataframe(
    sns.scatterplot,
    x="UMAP1_j",
    y="UMAP2_j",
    hue="Cancer",
    palette=sns.color_palette(
        "tab20",
        df["Cancer"].nunique()
    ),
    s=15,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.2
)

# =========================
# Styling
# =========================
g.set_axis_labels(
    "UMAP-1",
    "UMAP-2"
)

g.set_titles(
    col_template="{col_name}"
)

# =========================
# Remove duplicate legends
# =========================
for ax in g.axes.flat:

    legend = ax.get_legend()

    if legend is not None:
        legend.remove()

# =========================
# Single legend
# =========================
handles, labels = (
    g.axes[0]
    .get_legend_handles_labels()
)

g.fig.legend(
    handles,
    labels,
    loc="upper right",
    fontsize=9,
    frameon=True,
    ncol=1
)

# =========================
# Layout
# =========================
plt.subplots_adjust(
    top=0.97,
    right=0.88,
    hspace=0.25
)

plt.tight_layout()

# =========================
# Save figure
# =========================
save_path = (
    RESULT_DIR
    / "faceted_multiomics_umap_vertical.png"
)

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"✅ Saved figure to: {save_path}")

plt.show()

# Cancer-wise Structure

Create a multi-panel visualization of UMAP embeddings, where each subplot corresponds to a different cancer type. It combines scatter plots (sample-level structure) with kernel density estimation (population-level structure) to reveal both local and global patterns in the embedding space.

- clustering of sample types
- separation between biological groups
- heterogeneity within cancer types

In [ ]:
g = sns.FacetGrid(
    df,
    col="Cancer",
    col_wrap=4,
    height=3,
    sharex=True,
    sharey=True
)

g.map_dataframe(
    sns.scatterplot,
    x="UMAP1_j",
    y="UMAP2_j",
    hue="Type",
    palette="Set2",
    s=10,
    alpha=0.7,
    edgecolor="black",
    linewidth=0.2
)

g.map_dataframe(
    sns.kdeplot,
    x="UMAP1_j",
    y="UMAP2_j",
    fill=True,
    alpha=0.5,
    levels=5,
    cmap="Blues"
)

g.set_axis_labels("UMAP-1", "UMAP-2")
g.set_titles(col_template="{col_name}")

plt.tight_layout()

# ------------------------------------------------------
# Save figure
# ------------------------------------------------------

plt.savefig(
    RESULT_DIR / "umap_cancer_facetgrid.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Dimensionality Reduction

We are explicitly projecting high-dimensional data into a **2-dimensional latent space**.

---

## Mathematical formulation

Let the original data matrix be:

$$
\mathbf{X} \in \mathbb{R}^{n \times d}
\tag{1}
$$

where:
- $n$ = number of samples  
- $d$ = number of original features (high-dimensional space)  

Dimensionality reduction learns a mapping:

$$
f: \mathbb{R}^{d} \rightarrow \mathbb{R}^{k}
\tag{2}
$$

where:

$$
k = n_{\text{components}} = 2
\tag{3}
$$

Thus each sample $x_i$ is transformed as:

$$
\mathbf{z}_i = f(\mathbf{x}_i), \quad \mathbf{z}_i \in \mathbb{R}^{2}
\tag{4}
$$

and the full embedding becomes:

$$
\mathbf{Z} \in \mathbb{R}^{n \times 2}
\tag{5}
$$

---

## Geometric interpretation

Each sample is mapped from a high-dimensional space into a 2D plane:

$$
(x_{i1}, x_{i2}, \dots, x_{id}) \rightarrow (z_{i1}, z_{i2})
\tag{6}
$$

This allows:

- visualization of complex biological structure  
- clustering analysis in 2D space  
- detection of separability between classes (e.g., cancer types)  

---

### 2. Structure preservation objective

Methods like PCA aim to preserve variance:

$$
\max_{\mathbf{W} \in \mathbb{R}^{d \times 2}} \; \mathrm{Var}(\mathbf{XW})
\tag{8}
$$

while nonlinear methods (UMAP/t-SNE) preserve neighborhood structure.

---

## In UMAP / t-SNE context

For UMAP, the optimization becomes:

$$
\min_{\mathbf{Z} \in \mathbb{R}^{n \times 2}} \mathcal{L}(\mathbf{Z})
\tag{9}
$$

where $\mathbf{Z}$ is constrained to 2 dimensions.

---

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

# Gene-to-Dominant Cancer Assignment (Labeling Strategy)

Assign each gene to the cancer type in which it is most active, based on its feature values across cancer-specific columns.

In [ ]:
# assign each gene to cancer where it is most active
cancer_cols = [c.split("_")[-1] for c in X.columns]
cancer_cols = np.array(cancer_cols)

dominant_cancer_idx = np.argmax(X.values, axis=1)
gene_labels = cancer_cols[dominant_cancer_idx]

unique_labels = sorted(set(gene_labels))
color_map = {c: i for i, c in enumerate(unique_labels)}


plot_colors = [
    COLORS[color_map[g] % len(COLORS)]
    for g in gene_labels
]

## Gene-Level Visualization with Dominant Cancer Signal

Project high-dimensional gene features into a 2D embedding space using UMAP, and colors each gene according to its dominant cancer association.

---

## Input Embedding

Let the gene embedding matrix be:

$$
\mathbf{Z} \in \mathbb{R}^{G \times 2} \tag{1}
$$

where:
- \( G \) = number of genes  
- \( \mathbf{z}_g = (z_{g1}, z_{g2}) \) = 2D UMAP coordinates of gene \( g \)

This is obtained from a nonlinear mapping:

$$
f_{\text{UMAP}}: \mathbb{R}^{d} \rightarrow \mathbb{R}^{2} \tag{2}
$$

such that:

$$
\mathbf{Z} = f_{\text{UMAP}}(\mathbf{X}) \tag{3}
$$

---

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=plot_colors,
    s=5,              # small points (many genes!)
    alpha=0.6
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Gene-Level UMAP (Colored by Dominant Cancer Signal)")

plt.tight_layout()
plt.show()

# Multi-Omics Feature Distribution 

Compare the statistical distribution of four normalized omics feature types: mutation frequency, gene expression, copy number alteration, and miRNA-derived regulatory scores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# Prepare flattened values
# ===============================
mf_vals = MF_all.values.flatten()
ge_vals = GE_all.values.flatten()
cna_vals = CNA_all.values.flatten()
mirna_vals = mirna_all.values.flatten()

# remove invalid values
mf_vals = mf_vals[np.isfinite(mf_vals)]
ge_vals = ge_vals[np.isfinite(ge_vals)]
cna_vals = cna_vals[np.isfinite(cna_vals)]
mirna_vals = mirna_vals[np.isfinite(mirna_vals)]

# ===============================
# Combine into list
# ===============================
data = [mf_vals, ge_vals, cna_vals, mirna_vals]
labels = ["Mutation (MF)", "Expression (GE)", "CNA", "miRNA"]

# ===============================
# Plot
# ===============================
plt.figure(figsize=(10, 6))

bp = plt.boxplot(
    data,
    labels=labels,
    patch_artist=True,
    showfliers=False
)


for patch, color in zip(bp["boxes"], list(COLORS.values())[:4]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# median styling
for median in bp["medians"]:
    median.set_color("black")

plt.axhline(0, linestyle="--")

plt.title("Multi-Omics Feature Distribution")
plt.ylabel("Normalized Feature Value")
plt.xlabel("Omics Type")

plt.tight_layout()
plt.show()

## Kernel Density Estimation

---

For each omics type \( k \), the probability density function is estimated as:

$$
p_k(x) \approx \frac{1}{N_k h} \sum_{i=1}^{N_k} K\left(\frac{x - x_i}{h}\right) \tag{1}
$$

where:
- \( K(\cdot) \) is a smoothing kernel (typically Gaussian)  
- \( h \) is the bandwidth parameter controlling smoothness  
- \( N_k \) is the number of observations for omics type \( k \)  

---



In [ ]:
import seaborn as sns
import pandas as pd

# ===============================
# Build long-format dataframe
# ===============================
df_plot = pd.DataFrame({
    "value": np.concatenate([mf_vals, ge_vals, cna_vals, mirna_vals]),
    "omics": (
        ["MF"] * len(mf_vals) +
        ["GE"] * len(ge_vals) +
        ["CNA"] * len(cna_vals) +
        ["miRNA"] * len(mirna_vals)
    )
})

# ===============================
# Plot
# ===============================
plt.figure(figsize=(10, 6))

sns.violinplot(
    x="omics",
    y="value",
    data=df_plot,
    palette=list(COLORS.values())[:4],
    inner="quartile"
)

plt.axhline(0, linestyle="--")

plt.title("Multi-Omics Feature Distribution")
plt.xlabel("Omics Type")
plt.ylabel("Normalized Feature Value")

plt.tight_layout()
plt.show()

# Multi-Omics Density Comparison

In [ ]:
plt.figure(figsize=(10, 6))

for vals, label, color in zip(data, labels, list(COLORS.values())[:4]):
    sns.kdeplot(vals, label=label, color=color, fill=True, alpha=0.3)

plt.title("Multi-Omics Density Comparison")
plt.xlabel("Feature Value")
plt.ylabel("Density")

plt.legend()
plt.tight_layout()
plt.show()

## Average Mutation Signal per Cancer Type

Summarize the overall mutation burden across cancer types by computing the mean mutation score from the gene–cancer mutation matrix.

---

### 1. Mutation feature matrix

$$
\mathbf{X}^{\text{MF}} \in \mathbb{R}^{G \times C}
$$

where:
- \(G\) = number of genes  
- \(C\) = number of cancer types  
- \(X^{\text{MF}}_{g,c}\) = mutation score of gene \(g\) in cancer \(c\)

---

### 2. Cancer-level aggregation (mean mutation burden)

For each cancer type \(c\), the average mutation signal is computed as:

$$
\mu_c = \frac{1}{G} \sum_{g=1}^{G} X^{\text{MF}}_{g,c}
\tag{1}
$$

where:
- \(\mu_c\) = mean mutation burden of cancer \(c\)

This reduces the gene-level matrix into a cancer-level vector:

$$
\mathbf{X}^{\text{MF}} \in \mathbb{R}^{G \times C}
\;\;\rightarrow\;\;
\boldsymbol{\mu} \in \mathbb{R}^{C}
\tag{2}
$$

---

### 3. Ranking cancer types

Cancer types are ranked in descending order of mutation burden:

$$
c^{*} = \mathrm{argsort}(\mu_c,\ \text{descending})
\tag{3}
$$


- High values indicate strong mutation burden (genomic instability)
- Low values indicate weaker mutation activity

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ===============================
# MF_all: gene × cancer
# ===============================
mut = MF_all.copy()

# average mutation per cancer
mut_per_cancer = mut.mean(axis=0)

# ===============================
# Bar plot
# ===============================
plt.figure(figsize=(6,4))

mut_per_cancer.sort_values(
    ascending=False
).plot(
    kind="bar",
    width=0.4   
)

plt.title("")
plt.ylabel("Mean Mutation Score")
plt.xlabel("")

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

# Mutation vs Expression Spearman Correlation

Evaluation of the relationship between mutation burden and gene expression using a rank-based correlation measure, providing insight into monotonic dependencies between two omics modalities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# -----------------------------------
# Prepare data
# -----------------------------------
x = np.log1p(MF_all[mask])
y = np.log1p(GE_all[mask])

# Flatten in case they are DataFrames
x = x.values.flatten()
y = y.values.flatten()

# Remove NaNs (important!)
valid = (~np.isnan(x)) & (~np.isnan(y))
x = x[valid]
y = y[valid]

# -----------------------------------
# Compute correlation
# -----------------------------------
corr, _ = spearmanr(x, y)

# -----------------------------------
# Plot
# -----------------------------------
plt.figure(figsize=(6,5))

plt.scatter(
    x,
    y,
    s=5,
    alpha=0.5
)

plt.xlabel("log(1 + Mutation)")
plt.ylabel("log(1 + Expression)")
plt.title(f"Mutation vs Expression (Spearman r = {corr:.2f})")

plt.tight_layout()
plt.show()

# Gene-wise Mutation–Expression Correlation Distribution

This analysis quantifies how mutation and expression profiles are related at the level of individual genes by computing gene-specific Spearman correlations across cancer types.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# -----------------------------------
# Use pipeline outputs
# -----------------------------------
mut = MF_all
expr = GE_all

# -----------------------------------
# Align genes
# -----------------------------------
common_genes = mut.index.intersection(expr.index)

gene_corrs = []

for g in common_genes:
    m = mut.loc[g].values
    e = expr.loc[g].values

    # Remove NaNs
    valid = (~np.isnan(m)) & (~np.isnan(e))
    m = m[valid]
    e = e[valid]

    # Skip bad cases
    if len(m) < 3 or np.var(m) == 0 or np.var(e) == 0:
        continue

    r, _ = spearmanr(m, e)
    gene_corrs.append(r)

# -----------------------------------
# Plot
# -----------------------------------
plt.figure(figsize=(6,5))

plt.hist(gene_corrs, bins=50)

plt.title("Gene-wise Mutation–Expression Correlation")
plt.xlabel("Spearman r")
plt.ylabel("Number of genes")

# Optional: add mean line
plt.axvline(np.mean(gene_corrs), linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Per-Cancer Mutation–Expression Correlation Analysis

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

# -----------------------------------
# Align genes + cancers
# -----------------------------------
common_genes = MF_all.index.intersection(GE_all.index)
common_cancers = MF_all.columns.intersection(GE_all.columns)

mut = MF_all.loc[common_genes, common_cancers]
expr = GE_all.loc[common_genes, common_cancers]

# -----------------------------------
# Compute per-cancer correlations
# -----------------------------------
cancer_corrs = {}
cancer_pvals = {}

for c in common_cancers:
    m = mut[c].values
    e = expr[c].values

    # Remove NaNs
    valid = (~np.isnan(m)) & (~np.isnan(e))
    m, e = m[valid], e[valid]

    # Optional log transform (recommended)
    m = np.log1p(m)
    e = np.log1p(e)

    # Skip bad cases
    if len(m) < 10 or np.var(m) == 0 or np.var(e) == 0:
        continue

    r, p = spearmanr(m, e)

    cancer_corrs[c] = r
    cancer_pvals[c] = p

# Convert to DataFrame
corr_df = pd.DataFrame({
    "Cancer": list(cancer_corrs.keys()),
    "Spearman_r": list(cancer_corrs.values()),
    "p_value": [cancer_pvals[c] for c in cancer_corrs.keys()]
}).sort_values("Spearman_r", ascending=False)

print(corr_df.head())

# Per-Cancer Mutation–Expression Correlation (Bar Plot)

This visualization displays cancer-specific Spearman correlation coefficients between mutation burden and gene expression, summarizing cross-omics coupling across tumor types.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

# make bar width match pandas .plot(kind="bar")
plt.bar(
    corr_df["Cancer"],
    corr_df["Spearman_r"],
    width=0.4
)

plt.xticks(rotation=0)

plt.ylabel("Spearman Correlation")
plt.title("")

plt.axhline(
    0,
    linestyle='--'
)

plt.tight_layout()
plt.show()

# Distribution of Per-Cancer Mutation–Expression Correlations

Summarize the distribution of Spearman correlation coefficients computed across cancer types, providing a global view of how mutation–expression coupling varies across tumors.

In [ ]:
plt.figure(figsize=(6,5))

plt.hist(corr_df["Spearman_r"], bins=20)

plt.axvline(corr_df["Spearman_r"].mean(), linestyle='--')
plt.axvline(corr_df["Spearman_r"].median(), linestyle=':')

plt.title("Distribution of Per-Cancer Correlations")
plt.xlabel("Spearman r")
plt.ylabel("Number of cancers")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

def compute_per_cancer_corr(df1, df2, log_transform=True, min_points=10):
    """
    Compute per-cancer Spearman correlation between two omics matrices
    df1, df2: gene x cancer matrices
    """
    common_genes = df1.index.intersection(df2.index)
    common_cancers = df1.columns.intersection(df2.columns)

    A = df1.loc[common_genes, common_cancers]
    B = df2.loc[common_genes, common_cancers]

    corrs = {}
    pvals = {}

    for c in common_cancers:
        x = A[c].values
        y = B[c].values

        valid = (~np.isnan(x)) & (~np.isnan(y))
        x, y = x[valid], y[valid]

        if log_transform:
            x = np.log1p(x)
            y = np.log1p(y)

        if len(x) < min_points or np.var(x) == 0 or np.var(y) == 0:
            continue

        r, p = spearmanr(x, y)

        corrs[c] = r
        pvals[c] = p

    return pd.DataFrame({
        "Cancer": list(corrs.keys()),
        "Spearman_r": list(corrs.values()),
        "p_value": [pvals[c] for c in corrs.keys()]
    }).sort_values("Spearman_r", ascending=False)

In [ ]:
# Mutation vs Expression (you already did)
corr_mut_expr = compute_per_cancer_corr(MF_all, GE_all)

# CNA vs Expression
corr_cna_expr = compute_per_cancer_corr(CNA_all, GE_all)

# miRNA vs Expression
corr_mirna_expr = compute_per_cancer_corr(mirna_all, GE_all)

print("CNA vs Expression")
print(corr_cna_expr.head())

print("miRNA vs Expression")
print(corr_mirna_expr.head())

# Distribution of Cross-Omics–Expression Correlations

Compare the distributions of per-cancer Spearman correlation coefficients across multiple omics modalities with respect to gene expression.


In [ ]:
plt.figure(figsize=(6,5))

plt.hist(corr_mut_expr["Spearman_r"], bins=15, alpha=0.5, label="Mutation")
plt.hist(corr_cna_expr["Spearman_r"], bins=15, alpha=0.5, label="CNA")
plt.hist(corr_mirna_expr["Spearman_r"], bins=15, alpha=0.5, label="miRNA")

plt.xlabel("Spearman r")
plt.ylabel("Number of cancers")
plt.title("Distribution of Omics–Expression Correlations")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr
import numpy as np

# Align genes + cancers
common_genes = MF_all.index.intersection(GE_all.index)
common_cancers = MF_all.columns.intersection(GE_all.columns)

mut = MF_all.loc[common_genes, common_cancers]
expr = GE_all.loc[common_genes, common_cancers]

# Flatten
mut_vals = mut.to_numpy().flatten()
expr_vals = expr.to_numpy().flatten()

# Clean
valid = np.isfinite(mut_vals) & np.isfinite(expr_vals)
mut_vals = mut_vals[valid]
expr_vals = expr_vals[valid]

# Optional log transform
mut_vals = np.log1p(mut_vals)
expr_vals = np.log1p(expr_vals)

# Correlation
corr, pval = spearmanr(mut_vals, expr_vals)

print(f"Global Spearman correlation: {corr:.4f} (p = {pval:.2e})")

# Mutation Distribution per Cancer Type

Visualization of the distribution of mutation-derived features across genes for each cancer type using a log-transformed gene × cancer matrix.

---

## Input representation

Let the mutation matrix be:

$$
\mathbf{M} \in \mathbb{R}^{G \times C}
$$

where:
- \(G\) = number of genes  
- \(C\) = number of cancer types  
- \(M_{g,c}\) = mutation-derived score for gene \(g\) in cancer \(c\)

We restrict to shared gene and cancer spaces:

$$
\mathcal{G}' = \mathcal{G}_{\text{MF}} \cap \mathcal{G}_{\text{GE}}, \quad
\mathcal{C}' = \mathcal{C}_{\text{MF}} \cap \mathcal{C}_{\text{GE}}
$$

yielding aligned matrix:

$$
\mathbf{M}' \in \mathbb{R}^{|\mathcal{G}'| \times |\mathcal{C}'|}
$$

---

## 1. Log transformation

To reduce skewness and stabilize variance:

$$
M'_{g,c} \leftarrow \log(1 + M'_{g,c})
\tag{1}
$$

This reduces the influence of extreme mutation values.

---

## 2. Reshaping to long format

We convert the matrix into a set of observations:

$$
\mathcal{D} = \{(g, c, M'_{g,c})\}
\tag{2}
$$

Each entry corresponds to a gene–cancer pair.

Equivalently:

$$
\mathbf{M}' \rightarrow \mathcal{D}
\tag{3}
$$

---

## 3. Per-cancer distribution

For each cancer type \(c\), define:

$$
\mathcal{D}_c = \{ M'_{g,c} \mid g \in \mathcal{G}' \}
\tag{4}
$$

The boxplot summarizes:
- Median:
$$
\tilde{M}_c
$$

- Interquartile range:
$$
\mathrm{IQR}_c = Q_{75}(\mathcal{D}_c) - Q_{25}(\mathcal{D}_c)
$$

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------
# Align matrices first (important!)
# -----------------------------------
common_genes = MF_all.index.intersection(GE_all.index)
common_cancers = MF_all.columns.intersection(GE_all.columns)

mut = MF_all.loc[common_genes, common_cancers]

# -----------------------------------
# Optional: log transform (recommended)
# -----------------------------------
mut = np.log1p(mut)

# -----------------------------------
# Reshape to long format
# -----------------------------------
mut_long = mut.stack().reset_index()
mut_long.columns = ["Gene", "Cancer", "Value"]

# -----------------------------------
# Plot
# -----------------------------------
plt.figure(figsize=(12,6))

mut_long.boxplot(
    by="Cancer",
    column="Value",
    rot=45,
    grid=False
)

plt.title("Mutation Distribution per Cancer")
plt.suptitle("")  # remove pandas default title
plt.xlabel("Cancer Type")
plt.ylabel("log(1 + Mutation Value)")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================
# INPUT MATRICES (assumed loaded)
# ==============================
# MF_all: mutation frequency (genes x cancers)
# GE_all: gene expression (genes x cancers)
# CNA_all: copy number alteration (genes x cancers)
# mirna_all: miRNA expression (genes x cancers) [optional]

# ------------------------------
# Align all matrices
# ------------------------------
common_genes = MF_all.index
common_cancers = MF_all.columns

for mat in [GE_all, CNA_all]:
    common_genes = common_genes.intersection(mat.index)
    common_cancers = common_cancers.intersection(mat.columns)

if 'mirna_all' in globals():
    common_genes = common_genes.intersection(mirna_all.index)
    common_cancers = common_cancers.intersection(mirna_all.columns)

MF = MF_all.loc[common_genes, common_cancers]
GE = GE_all.loc[common_genes, common_cancers]
CNA = CNA_all.loc[common_genes, common_cancers]

if 'mirna_all' in globals():
    MIRNA = mirna_all.loc[common_genes, common_cancers]

# ------------------------------
# Transformations
# ------------------------------
MF_log = np.log1p(MF)

GE_z = (
    GE - GE.mean()
) / (GE.std() + 1e-8)

# Signed log transform for CNA
CNA_trans = np.sign(CNA) * np.log1p(np.abs(CNA))

if 'mirna_all' in globals():
    MIRNA_log = np.log1p(MIRNA)

# ------------------------------
# Helper: reshape to long format
# ------------------------------
def to_long(df, name):
    tmp = df.stack().reset_index()
    tmp.columns = ['Gene', 'Cancer', name]
    return tmp

mf_long = to_long(MF_log, 'MF')
ge_long = to_long(GE_z, 'GE')
cna_long = to_long(CNA_trans, 'CNA')

if 'mirna_all' in globals():
    mirna_long = to_long(MIRNA_log, 'miRNA')

# ------------------------------
# CNA amplification/deletion
# ------------------------------
amp_freq = (
    (CNA > 0).sum(axis=0)
    / CNA.shape[0]
)

del_freq = (
    (CNA < 0).sum(axis=0)
    / CNA.shape[0]
)

# ------------------------------
# Plotting
# ------------------------------
fig = plt.figure(figsize=(18, 14))

# ==================================================
# Panel A: Mutation Frequency
# ==================================================
ax1 = plt.subplot(2, 2, 1)

mf_long.boxplot(
    by='Cancer',
    column='MF',
    ax=ax1,
    rot=45,
    grid=False
)

ax1.set_title('Mutation Frequency (log1p)')
ax1.set_xlabel('')   # remove Cancer Type
ax1.set_ylabel('log(1 + MF)')

# ==================================================
# Panel B: Gene Expression
# ==================================================
ax2 = plt.subplot(2, 2, 2)

ge_long.boxplot(
    by='Cancer',
    column='GE',
    ax=ax2,
    rot=45,
    grid=False
)

ax2.set_title('Gene Expression (Z-score)')
ax2.set_xlabel('')   # remove Cancer Type
ax2.set_ylabel('Z-score')

# ==================================================
# Panel C: CNA Alterations
# ==================================================
ax3 = plt.subplot(2, 2, 3)

x = np.arange(len(common_cancers))

ax3.bar(
    x - 0.2,
    amp_freq.values,
    width=0.4,
    label='Amplification'
)

ax3.bar(
    x + 0.2,
    del_freq.values,
    width=0.4,
    label='Deletion'
)

ax3.set_xticks(x)

ax3.set_xticklabels(
    common_cancers,
    rotation=45
)

ax3.set_title('CNA Alteration Frequency')

ax3.set_xlabel('')   # remove Cancer Type
ax3.set_ylabel('Fraction of Genes')

ax3.legend()

# ==================================================
# Panel D: miRNA
# ==================================================
ax4 = plt.subplot(2, 2, 4)

if 'mirna_all' in globals():

    mirna_long.boxplot(
        by='Cancer',
        column='miRNA',
        ax=ax4,
        rot=45,
        grid=False
    )

    ax4.set_title('miRNA Expression (log1p)')
    ax4.set_xlabel('')   # remove Cancer Type
    ax4.set_ylabel('log(1 + expression)')

else:

    ax4.text(
        0.5,
        0.5,
        'miRNA not provided',
        ha='center',
        va='center'
    )

    ax4.set_title('miRNA (optional)')
    ax4.axis('off')

# ------------------------------
# Remove automatic pandas titles
# ------------------------------
plt.suptitle('')

# ------------------------------
# Layout
# ------------------------------
plt.tight_layout()

# ------------------------------
# Save PNG
# ------------------------------
plt.savefig(
    RESULT_DIR / 'multiomics_panel.png',
    dpi=300,
    bbox_inches='tight'
)

# ------------------------------
# Show
# ------------------------------
plt.show()

print("Saved figure: multiomics_panel.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================
# ALIGN MATRICES
# ==============================
common_genes = MF_all.index
common_cancers = MF_all.columns

for mat in [GE_all]:
    common_genes = common_genes.intersection(mat.index)
    common_cancers = common_cancers.intersection(mat.columns)

if 'mirna_all' in globals():
    common_genes = common_genes.intersection(mirna_all.index)
    common_cancers = common_cancers.intersection(mirna_all.columns)

MF = MF_all.loc[common_genes, common_cancers]
GE = GE_all.loc[common_genes, common_cancers]

if 'mirna_all' in globals():
    MIRNA = mirna_all.loc[common_genes, common_cancers]

# ==============================
# TRANSFORMATIONS
# ==============================
MF_log = np.log1p(MF)

GE_z = (
    GE - GE.mean()
) / (GE.std() + 1e-8)

if 'mirna_all' in globals():
    MIRNA_log = np.log1p(MIRNA)

# ==============================
# HELPER
# ==============================
def to_long(df, name):
    tmp = df.stack().reset_index()
    tmp.columns = ['Gene', 'Cancer', name]
    return tmp

mf_long = to_long(MF_log, 'MF')
ge_long = to_long(GE_z, 'GE')

if 'mirna_all' in globals():
    mirna_long = to_long(MIRNA_log, 'miRNA')

# ==============================
# PLOTTING
# ==============================
fig = plt.figure(figsize=(18, 4))

# ==================================================
# Panel A: Gene Expression
# ==================================================
ax1 = plt.subplot(1, 3, 1)

ge_long.boxplot(
    by='Cancer',
    column='GE',
    ax=ax1,
    rot=0,
    grid=False
)

ax1.set_title('Gene Expression (Z-score)')
ax1.set_xlabel('')
ax1.set_ylabel('Z-score')

# ==================================================
# Panel B: Mutation Frequency
# ==================================================
ax2 = plt.subplot(1, 3, 2)

mf_long.boxplot(
    by='Cancer',
    column='MF',
    ax=ax2,
    rot=0,
    grid=False
)

ax2.set_title('Mutation Frequency (log1p)')
ax2.set_xlabel('')
ax2.set_ylabel('log(1 + MF)')

# ==================================================
# Panel C: miRNA
# ==================================================
ax3 = plt.subplot(1, 3, 3)

if 'mirna_all' in globals():

    mirna_long.boxplot(
        by='Cancer',
        column='miRNA',
        ax=ax3,
        rot=0,
        grid=False
    )

    ax3.set_title('miRNA Expression (log1p)')
    ax3.set_xlabel('')
    ax3.set_ylabel('log(1 + expression)')

else:

    ax3.text(
        0.5,
        0.5,
        'miRNA not provided',
        ha='center',
        va='center'
    )

    ax3.set_title('miRNA (optional)')
    ax3.axis('off')

# ==============================
# CLEANUP
# ==============================
plt.suptitle('')

plt.subplots_adjust(
    wspace=0.45,   # horizontal space between plots
    hspace=0.3     # vertical space (future-safe)
)

# ==============================
# SAVE
# ==============================
plt.savefig(
    RESULT_DIR / 'multiomics_panel_no_cna.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved figure: multiomics_panel_no_cna.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================
# ALIGN MATRICES
# ==============================
common_genes = MF_all.index
common_cancers = MF_all.columns

for mat in [GE_all]:
    common_genes = common_genes.intersection(mat.index)
    common_cancers = common_cancers.intersection(mat.columns)

if 'mirna_all' in globals():
    common_genes = common_genes.intersection(mirna_all.index)
    common_cancers = common_cancers.intersection(mirna_all.columns)

# ------------------------------
# ORDER: MF -> MIRNA -> GE
# ------------------------------
MF = MF_all.loc[common_genes, common_cancers]

if 'mirna_all' in globals():
    MIRNA = mirna_all.loc[common_genes, common_cancers]

GE = GE_all.loc[common_genes, common_cancers]

# ==============================
# TRANSFORMATIONS
# ==============================
MF_log = np.log1p(MF)

if 'mirna_all' in globals():
    MIRNA_log = np.log1p(MIRNA)

GE_z = (
    GE - GE.mean()
) / (GE.std() + 1e-8)

# ==============================
# HELPER
# ==============================
def to_long(df, name):
    tmp = df.stack().reset_index()
    tmp.columns = ['Gene', 'Cancer', name]
    return tmp

mf_long = to_long(MF_log, 'MF')

if 'mirna_all' in globals():
    mirna_long = to_long(MIRNA_log, 'miRNA')

ge_long = to_long(GE_z, 'GE')

# ==============================
# PLOTTING
# ==============================
fig = plt.figure(figsize=(20, 4))

# ==================================================
# Panel A: Mutation Frequency (MF)
# ==================================================
ax1 = plt.subplot(1, 3, 1)

mf_long.boxplot(
    by='Cancer',
    column='MF',
    ax=ax1,
    rot=0,
    grid=False
)

ax1.set_title('Mutation Frequency (log1p)')
ax1.set_xlabel('')
ax1.set_ylabel('log(1 + MF)')

# ==================================================
# Panel B: miRNA
# ==================================================
ax2 = plt.subplot(1, 3, 2)

if 'mirna_all' in globals():

    mirna_long.boxplot(
        by='Cancer',
        column='miRNA',
        ax=ax2,
        rot=0,
        grid=False
    )

    ax2.set_title('miRNA Expression (log1p)')
    ax2.set_xlabel('')
    ax2.set_ylabel('log(1 + expression)')

else:

    ax2.text(
        0.5,
        0.5,
        'miRNA not provided',
        ha='center',
        va='center'
    )

    ax2.set_title('miRNA (optional)')
    ax2.axis('off')

# ==================================================
# Panel C: Gene Expression (GE)
# ==================================================
ax3 = plt.subplot(1, 3, 3)

ge_long.boxplot(
    by='Cancer',
    column='GE',
    ax=ax3,
    rot=0,
    grid=False
)

ax3.set_title('Gene Expression (Z-score)')
ax3.set_xlabel('')
ax3.set_ylabel('Z-score')

# ==============================
# CLEANUP
# ==============================
plt.suptitle('')

plt.subplots_adjust(
    wspace=0.45,
    hspace=0.3
)

# ==============================
# SAVE
# ==============================
plt.savefig(
    RESULT_DIR / 'multiomics_panel_no_cna.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved figure: multiomics_panel_no_cna.png")

# Mutation Distribution per Cancer Type (Violin Plot Analysis)

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,6))

sns.violinplot(
    data=mut_long,
    x="Cancer",
    y="Value",
    cut=0
)

plt.xticks(rotation=0)

# remove title
plt.title("")

# remove x-axis label
plt.xlabel("")

plt.ylabel("log(1 + Mutation Value)")

plt.tight_layout()

plt.savefig(
    RESULT_DIR / "mutation_violin_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Multi-Omics Distributions with Log-Scaled Frequency

Display the distributions of multi-omics features using both value transformation and logarithmic scaling of histogram frequencies.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# Prepare matrices
# ===============================
mut = MF_all.copy()
expr = GE_all.copy()
cna = CNA_all.copy()
mirna = mirna_all.copy()

# ===============================
# Helper: clean + flatten
# ===============================
def clean_flatten(df):
    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]
    return vals

mut_vals = clean_flatten(mut)
expr_vals = clean_flatten(expr)
cna_vals = clean_flatten(cna)
mirna_vals = clean_flatten(mirna)

# ===============================
# Safe log1p transform (for values)
# ===============================
def safe_log1p(x):
    x = np.maximum(x, 0)
    return np.log1p(x)

mut_log = safe_log1p(mut_vals)
expr_log = safe_log1p(expr_vals)
mirna_log = safe_log1p(mirna_vals)

# CNA not log-transformed
cna_clean = cna_vals

# ===============================
# Plot with LOG FREQUENCY
# ===============================
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Mutation
axes[0, 0].hist(mut_log, bins=100, log=True)
axes[0, 0].set_title("Mutation (log1p)")

# Expression
axes[0, 1].hist(expr_log, bins=100, log=True)
axes[0, 1].set_title("Expression (log1p)")

# CNA
axes[1, 0].hist(cna_clean, bins=100, log=True)
axes[1, 0].set_title("CNA")

# miRNA
axes[1, 1].hist(mirna_log, bins=100, log=True)
axes[1, 1].set_title("miRNA (log1p)")

# Axis labels
for ax in axes.flat:
    ax.set_xlabel("Value")
    ax.set_ylabel("Log Frequency")

plt.suptitle("Multi-Omics Signal Distributions (Log Frequency)", fontsize=14)
plt.tight_layout()
plt.show()

## Multi-Omics Signal Distribution Visualization

The histograms visualize the global distribution of integrated omics signals across all genes and cancers after preprocessing and normalization.

Let

$$
X^{(m)} = \{x_1, x_2, \dots, x_n\}
$$

represent the feature values for omics modality $m$, where:

$$
m \in \{\text{Mutation}, \text{Expression}, \text{miRNA}\}
$$

The raw values are flattened across all genes and cancer types:

$$
X^{(m)}_{\text{flat}} =
\mathrm{vec}\left(
\mathbf{M}^{(m)}
\right)
$$

where:

$$
\mathbf{M}^{(m)} \in \mathbb{R}^{G \times C}
$$

with:

- $G$ = number of genes
- $C$ = number of cancers

Non-finite values are removed using:

$$
X^{(m)}_{\text{clean}}
=
\left\{
x \in X^{(m)}_{\text{flat}}
\; | \;
x \in \mathbb{R}
\right\}
$$

To stabilize highly skewed biological distributions, a logarithmic transformation is applied:

$$
x' = \log(1 + \max(x,0))
$$

which compresses extreme values while preserving relative variation.

The histograms are plotted using logarithmic frequency scaling:

$$
y = \log(\text{count})
$$

allowing simultaneous visualization of both dense and sparse regions of the distributions.

Mutation distributions typically exhibit strong sparsity with heavy-tailed behavior due to rare but highly recurrent driver mutations.

Expression distributions are generally smoother and denser because transcriptomic measurements vary continuously across samples.

miRNA distributions often show intermediate sparsity, reflecting selective regulatory activation across cancer types.

The resulting plots provide a sanity check for:

$$
\text{Normalization}
\rightarrow
\text{Signal range}
\rightarrow
\text{Distribution balance}
\rightarrow
\text{Cross-omics compatibility}
$$

These distributions are important before downstream analyses such as:

$$
\text{UMAP}
\rightarrow
\text{Clustering}
\rightarrow
\text{Graph Neural Networks}
\rightarrow
\text{Cancer Driver Prediction}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# Prepare matrices
# ===============================
expr = GE_all.copy()
mut = MF_all.copy()
mirna = mirna_all.copy()

# ===============================
# Helper: clean + flatten
# ===============================
def clean_flatten(df):
    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]
    return vals

expr_vals = clean_flatten(expr)
mut_vals = clean_flatten(mut)
mirna_vals = clean_flatten(mirna)

# ===============================
# Safe log1p transform
# ===============================
def safe_log1p(x):
    x = np.maximum(x, 0)
    return np.log1p(x)

expr_log = safe_log1p(expr_vals)
mut_log = safe_log1p(mut_vals)
mirna_log = safe_log1p(mirna_vals)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# -------------------------------
# Expression
# -------------------------------
axes[0].hist(expr_log, bins=100, log=True)
axes[0].set_title("Expression (log1p)", fontsize=13)

# -------------------------------
# Mutation
# -------------------------------
axes[1].hist(mut_log, bins=100, log=True)
axes[1].set_title("Mutation (log1p)", fontsize=13)

# -------------------------------
# miRNA
# -------------------------------
axes[2].hist(mirna_log, bins=100, log=True)
axes[2].set_title("miRNA (log1p)", fontsize=13)

# -------------------------------
# Labels
# -------------------------------
for ax in axes:
    ax.set_xlabel("", fontsize=11)
    ax.set_ylabel("Log Frequency", fontsize=11)

plt.suptitle(
    "",
    fontsize=16
)

plt.tight_layout()

# ===============================
# Save PNG
# ===============================
plt.savefig(
    RESULT_DIR / "multiomics_distributions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("✅ Saved: multiomics_distributions.png")

# miRNA Signal Distribution per Cancer

This analysis visualizes the distribution of miRNA-derived signals separately for each cancer type using logarithmic scaling of frequency.

---

## Input representation

Let the miRNA feature matrix be:

$$
\mathbf{R} \in \mathbb{R}^{G \times C}
$$

where:
- \(G\) = number of genes  
- \(C\) = number of cancer types  
- \(R_{g,c}\) = aggregated miRNA signal for gene \(g\) in cancer \(c\)

---

## 1. Per-cancer extraction

For each cancer \(c \in \mathcal{C}\), we extract:

$$
\mathbf{r}_c = (R_{1,c}, R_{2,c}, \dots, R_{G,c})
\tag{1}
$$

Invalid values are removed:

$$
\mathbf{r}_c \leftarrow \{r_i \mid r_i \in \mathbb{R}, \; \text{finite}\}
\tag{2}
$$

---

## 2. Histogram estimation

We estimate the empirical distribution:

$$
p_c(x) = \mathrm{Histogram}(\mathbf{r}_c)
\tag{3}
$$

Each bin count is:

$$
y_j = \#\{r_i \in \text{bin}_j\}
\tag{4}
$$

---

## 3. Log-frequency scaling

To better visualize heavy-tailed distributions:

$$
y_j^{*} = \log(y_j)
\tag{5}
$$

implemented in practice via:

```python
plt.yscale("log")
```

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


mirna_df = mirna_all.copy()

print("miRNA matrix shape:", mirna_df.shape)

# ===============================
# Flatten values
# ===============================
mirna_cancer_vals = mirna_df.values.flatten()

# remove NaN / inf
mirna_cancer_vals = mirna_cancer_vals[
    np.isfinite(mirna_cancer_vals)
]

print("Total values:", len(mirna_cancer_vals))



# ===============================
# Plot each cancer separately
# ===============================
for c in mirna_df.columns:
    plt.figure(figsize=(6, 4))

    vals = mirna_df[c].values
    vals = vals[np.isfinite(vals)]

    plt.hist(
        vals,
        bins=100
    )

    # log frequency
    plt.yscale("log")

    plt.title(f"miRNA Distribution: {c}")
    plt.xlabel("miRNA Signal")
    plt.ylabel("Frequency (log)")

    plt.tight_layout()
    plt.show()


## miRNA-Derived Gene Signal Distribution

We visualize the distribution of miRNA-derived gene regulatory signals across cancer types.

Each value represents the aggregated regulatory influence of miRNAs on genes.

---

### Mathematical Formulation

Let the miRNA-derived signal for gene $g$ in cancer $c$ be:

$$
x_{g,c} \in \mathbb{R}
\tag{1}
$$

The full dataset is:

$$
\mathbf{X} \in \mathbb{R}^{G \times C}
\tag{2}
$$

where:
- $G$ = number of genes  
- $C$ = number of cancer types  

---

### Distribution Analysis

For each cancer $c$, we analyze:

$$
\{ x_{g,c} \mid g = 1, \dots, G \}
\tag{3}
$$

To better visualize heavy-tailed distributions, we apply log scaling:

$$
y = \log(\text{frequency})
\tag{4}
$$

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6, 4))

# increase global font size
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10
})

# consistent color mapping
cancers = list(mirna_df.columns)
color_map = {
    c: COLORS[i % len(COLORS)]
    for i, c in enumerate(cancers)
}

for c in cancers:
    vals = mirna_df[c].values
    vals = vals[np.isfinite(vals)]

    plt.hist(
        vals,
        bins=100,
        alpha=0.6,
        color=color_map[c],
        label=c
    )

plt.yscale("log")

plt.title("")
plt.xlabel("miRNA-derived Gene Signal")
plt.ylabel("Frequency (log scale)")

# plt.legend(
#     bbox_to_anchor=(1.02, 1),
#     loc="upper left",
#     ncol=1
# )


plt.legend(
    loc="upper right",
    frameon=True,
    fontsize=10
)

plt.tight_layout()

# --------------------------------
# Save figure
# --------------------------------
save_path = RESULT_DIR / "mirna_signal_distribution_per_cancer.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"✅ Saved figure to: {save_path}")
plt.show()

## 📊 miRNA-Derived Gene Signal Distribution

This visualization shows the **distribution of miRNA-derived gene regulatory signals** across different cancer types using Kernel Density Estimation (KDE).

---

## 1. Input Representation

Let:

- \( \mathcal{G} \) = set of genes  
- \( \mathcal{C} \) = set of cancers  
- \( x_{g,c} \) = miRNA-derived signal for gene \( g \) in cancer \( c \)

Each column of the matrix:

$$
X^{(\text{miRNA})} = \{ x_{g,c} \} \tag{1}
$$

represents the gene-level miRNA regulatory signal for a specific cancer.

---

## 2. Data Cleaning

Invalid values are removed before density estimation:

$$
\tilde{x}_{g,c} = x_{g,c} \quad \text{such that } x_{g,c} \in \mathbb{R},\ \text{finite} \tag{2}
$$

---

## 3. Kernel Density Estimation (KDE)

For each cancer \( c \), we estimate the probability density function:

$$
\hat{f}_c(x) =
\frac{1}{n h} \sum_{i=1}^{n}
K\left( \frac{x - x_i}{h} \right) \tag{3}
$$

where:
- \( K(\cdot) \) is a kernel function (Gaussian in seaborn),
- \( h \) is the bandwidth,
- \( n \) is the number of genes.

---

## 4. Visualization

Each curve represents:

$$
\hat{f}_c(x) \quad \forall c \in \mathcal{C} \tag{4}
$$

- Different colors correspond to different cancer types  
- The x-axis represents miRNA-derived gene signal  
- The y-axis represents estimated density  

---

In [ ]:
import seaborn as sns

plt.figure(figsize=(6, 4))

for c in cancers:
    vals = mirna_df[c].values
    vals = vals[np.isfinite(vals)]

    sns.kdeplot(
        vals,
        label=c,
        color=color_map[c],
        linewidth=2
    )

plt.title("")
plt.xlabel("miRNA-derived Gene Signal")
plt.ylabel("Density")

plt.legend(
    loc="upper right",
    frameon=True,
    fontsize=10
)

plt.tight_layout()

# --------------------------------
# Save figure
# --------------------------------
save_path = RESULT_DIR / "mirna_signal_density_per_cancer.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"✅ Saved figure to: {save_path}")
plt.show()

## Multi-Omics Feature Distribution Comparison

We compare the distributions of four omics-derived gene features:

- Mutation Frequency (MF)  
- Gene Expression (GE)  
- Copy Number Alteration (CNA)  
- miRNA-derived regulation (miRNA)  

---

### Mathematical Formulation

Each omics layer produces a gene-level signal:

$$
x^{(m)}_{g,c}
\tag{1}
$$

where:
- $m \in \{\text{MF}, \text{GE}, \text{CNA}, \text{miRNA}\}$
- $g$ = gene
- $c$ = cancer type

---

### Distribution Representation

For each omics type:

$$
P(x^{(m)})
\tag{2}
$$

is visualized as a histogram over all genes and cancers.

---

### Interpretation

This comparison highlights:

- Scale differences across omics layers  
- Sparsity (MF) vs continuous signals (GE, miRNA)  
- Structural variation (CNA)  
- Regulatory dispersion (miRNA)  

---

### Summary

Each subplot represents:

$$
\{ x^{(m)}_{g,c} \}_{g,c}
\tag{3}
$$

allowing direct comparison of feature distributions across modalities.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ===============================
# Prepare data
# ===============================
data_dict = {
    "Mutation (MF)": MF_all,
    "Expression (GE)": GE_all,
    "CNA": CNA_all,
    "miRNA": mirna_all
}

# ===============================
# Plot
# ===============================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# global font scaling
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14
})

axes = axes.flatten()

for idx, (title, df) in enumerate(data_dict.items()):
    ax = axes[idx]

    vals = df.values.flatten()
    vals = vals[np.isfinite(vals)]

    ax.hist(
        vals,
        bins=100,
        alpha=0.7,
        color=COLORS[idx % len(COLORS)]
    )

    # log scale improves visibility
    ax.set_yscale("log")

    ax.set_title(title)
    ax.set_xlabel("Feature Value")
    ax.set_ylabel("Frequency (log)")

# layout
plt.suptitle("Multi-Omics Feature Distribution Comparison", fontsize=18)
plt.tight_layout()
plt.show()

# Gene-Level Clustering, UMAP Embedding, and Functional Enrichment

Unsupervised learning on gene-level multi-omics features, followed by functional enrichment analysis to interpret biological modules.

---

## 1. Data cleaning

Let the input matrix be:

$$
\mathbf{X} \in \mathbb{R}^{G \times F}
$$

where:
- \(G\) = number of genes  
- \(F\) = number of multi-omics features across cancers  

We remove invalid entries:

$$
\mathbf{X}_{\text{safe}} = \{x_{ij} \mid x_{ij} \in \mathbb{R}, \; \text{finite}\}
$$

ensuring numerical stability.

---

## 2. Feature scaling

We standardize each feature:

$$
x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
\tag{1}
$$

where:
- \(\mu_j\) = mean of feature \(j\)  
- \(\sigma_j\) = standard deviation of feature \(j\)

This yields:

$$
\mathbf{X}_{\text{scaled}} \in \mathbb{R}^{G \times F}
\tag{2}
$$

---

## 3. UMAP dimensionality reduction

We project high-dimensional features into 2D:

$$
\mathbf{Z} = \mathrm{UMAP}(\mathbf{X}_{\text{scaled}})
\tag{3}
$$

$$
\mathbf{Z} \in \mathbb{R}^{G \times 2}
$$

UMAP preserves local structure by minimizing:

$$
\min_{\mathbf{Z}} \; \mathrm{KL}(P \parallel Q)
\tag{4}
$$

where:
- \(P\) = high-dimensional neighbor graph  
- \(Q\) = low-dimensional embedding  

Key parameters:
- \(n_{\text{neighbors}} = 20\)
- \(\text{min\_dist} = 0.15\)

---

## 4. K-means clustering

We partition genes into \(k = 8\) clusters:

$$
\mathcal{C} = \{C_1, C_2, \dots, C_k\}
\tag{5}
$$

by minimizing:

$$
\sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2
\tag{6}
$$

where:
- \(\mu_i\) = centroid of cluster \(i\)

---

## 5. Cluster assignment

Each gene \(g\) is assigned:

$$
\text{Cluster}(g) \in \{1, \dots, k\}
\tag{7}
$$

forming:

$$
\mathcal{D} = \{(g, c_g)\}
\tag{8}
$$

---

## 6. UMAP visualization

Each gene is mapped as:

$$
g \rightarrow (z_{g,1}, z_{g,2})
\tag{9}
$$

and colored by cluster:

$$
\text{color}(g) = \text{Cluster}(g)
\tag{10}
$$

This reveals geometric separation of gene modules.

---

## 7. Functional enrichment analysis

For each cluster \(C_i\), we evaluate enrichment:

$$
\text{Enrichment}(C_i, T)
\tag{11}
$$

for biological terms \(T\) from:
- Gene Ontology (GO:BP)
- KEGG pathways  
- Reactome (REAC)  
- Human Phenotype Ontology (HP)

---

## 8. Statistical testing

Enrichment is evaluated using a hypergeometric model:

$$
p = P(X \geq k)
\tag{12}
$$

where:
- \(X\) = overlap between cluster genes and term genes  
- \(k\) = observed overlap  

---

## 9. Significance filtering

We retain terms satisfying:

$$
p < 0.05
\tag{13}
$$

and:

$$
\text{intersection\_size} \geq 3
\tag{14}
$$

---

## 10. Top pathway selection

For each cluster:

$$
T_i = \text{Top-}N \text{ terms ranked by } p
\tag{15}
$$

We transform:

$$
s = -\log_{10}(p)
\tag{16}
$$

---

## 11. Dot plot visualization

Each point represents:

$$
(\text{Cluster}, \text{Term})
\tag{17}
$$

with:
- size ∝ \( -\log_{10}(p) \)  
- color = pathway source  

---

## 12. Heatmap representation

We define:

$$
H_{t,i} = -\log_{10}(p_{t,i})
\tag{18}
$$

forming:

$$
\mathbf{H} \in \mathbb{R}^{T \times k}
\tag{19}
$$

where:
- \(t\) = biological term  
- \(i\) = cluster  

---

## 13. Source-specific heatmaps

We construct:

$$
\mathbf{H}^{(\text{GO})}, \mathbf{H}^{(\text{KEGG})}, \mathbf{H}^{(\text{REAC})}, \mathbf{H}^{(\text{HP})}
\tag{20}
$$

to improve interpretability and biological resolution.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import umap
from gprofiler import GProfiler

# =========================================================
# 1. Clean gene matrix
# =========================================================
X_safe = X.replace([np.inf, -np.inf], np.nan).dropna()
gene_names = X_safe.index

print("[INFO] Clean matrix:", X_safe.shape)

# =========================================================
# 2. Scale
# =========================================================
X_scaled = StandardScaler().fit_transform(X_safe)

# =========================================================
# 3. UMAP
# =========================================================
reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.15,
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

# =========================================================
# 4. Clustering
# =========================================================
k = 8
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

# =========================================================
# 5. Cluster DataFrame
# =========================================================
cluster_df = pd.DataFrame({
    "Gene": gene_names,
    "Cluster": cluster_labels
})

print("[INFO] Clusters created:", cluster_df.shape)

# =========================================================
# 6. UMAP Plot (clusters)
# =========================================================
# if isinstance(COLORS, dict):
#     COLORS_LIST = list(COLORS.values())
# else:
#     COLORS_LIST = COLORS

# umap_colors = [
#     COLORS_LIST[c % len(COLORS_LIST)]
#     for c in cluster_labels
# ]

# plt.figure(figsize=(6,5))

# plt.scatter(
#     X_umap[:,0],
#     X_umap[:,1],
#     c=umap_colors,
#     s=6,
#     alpha=0.8
# )

# plt.title("Gene UMAP (Clusters)")
# plt.xlabel("UMAP1")
# plt.ylabel("UMAP2")

# plt.tight_layout()
# plt.show()
# =========================================================
# 6. UMAP Plot (clusters)
# =========================================================
if isinstance(COLORS, dict):
    COLORS_LIST = list(COLORS.values())
else:
    COLORS_LIST = COLORS

umap_colors = [
    COLORS_LIST[c % len(COLORS_LIST)]
    for c in cluster_labels
]

plt.figure(figsize=(6,5))

plt.scatter(
    X_umap[:,0],
    X_umap[:,1],
    c=umap_colors,
    s=6,
    alpha=0.8
)

# -----------------------------
# Cluster legend
# -----------------------------
handles = []

for cid in sorted(np.unique(cluster_labels)):

    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker='o',
            color='w',
            label=f'Cluster {cid}',
            markerfacecolor=COLORS_LIST[cid % len(COLORS_LIST)],
            markersize=8
        )
    )

plt.legend(
    handles=handles,
    loc="lower right",
    fontsize=9,
    frameon=True
)

plt.title("Gene UMAP (Clusters)")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.tight_layout()

# -----------------------------
# Save
# -----------------------------
save_path = RESULT_DIR / "gene_umap_clusters.png"

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"✅ Saved: {save_path}")

plt.show()
# =========================================================
# 7. Enrichment (GO + KEGG + Reactome + HPO)
# =========================================================
gp = GProfiler(return_dataframe=True)

enrichment_results = []

for cluster_id in sorted(cluster_df["Cluster"].unique()):

    genes = cluster_df.loc[
        cluster_df["Cluster"] == cluster_id, "Gene"
    ].tolist()

    if len(genes) < 10:
        print(f"[SKIP] Cluster {cluster_id} too small")
        continue

    print(f"[RUN] Cluster {cluster_id} ({len(genes)} genes)")

    res = gp.profile(
        organism="hsapiens",
        query=genes,
        sources=[
            "GO:BP",
            "KEGG",
            "REAC",   # Reactome
            "HP"      # Human Phenotype Ontology
        ]
    )

    if res is None or res.empty:
        print(f"[WARN] No results for cluster {cluster_id}")
        continue

    res["Cluster"] = cluster_id
    enrichment_results.append(res)

# =========================================================
# 8. Combine results
# =========================================================
if len(enrichment_results) == 0:
    raise ValueError("No enrichment results found")

enrich_df = pd.concat(enrichment_results, ignore_index=True)

print("[INFO] Total enrichment rows:", enrich_df.shape)

# =========================================================
# 9. Filter significant
# =========================================================
enrich_df = enrich_df[enrich_df["p_value"] < 0.05].copy()

# robustness filter (recommended)
enrich_df = enrich_df[enrich_df["intersection_size"] >= 3]

# =========================================================
# 10. Select top pathways per cluster
# =========================================================
TOP_N = 5

enrich_top = (
    enrich_df.sort_values("p_value")
    .groupby("Cluster")
    .head(TOP_N)
    .copy()
)

enrich_top["-log10(p)"] = -np.log10(enrich_top["p_value"])

# =========================================================
# 11. DOT PLOT (colored by source)
# =========================================================
source_palette = {
    "GO:BP": "#1f77b4",
    "KEGG": "#ff7f0e",
    "REAC": "#2ca02c",
    "HP": "#d62728"
}

plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    palette=source_palette,
    sizes=(50, 300)
)

# plt.title("Cluster → Pathway / Phenotype Enrichment")
plt.title("")
plt.xlabel("Cluster")
plt.ylabel("Term")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    title="Source"
)

plt.tight_layout()
plt.show()

# =========================================================
# 12. COMBINED HEATMAP
# =========================================================
heatmap_data = enrich_top.pivot_table(
    index="name",
    columns="Cluster",
    values="p_value",
    aggfunc="min"
)

heatmap_data = -np.log10(heatmap_data)

plt.figure(figsize=(10, 10))

sns.heatmap(
    heatmap_data,
    cmap="viridis",
    linewidths=0.5
)

plt.title("Pathway Enrichment Heatmap (-log10 p-value)")
plt.xlabel("Cluster")
plt.ylabel("Term")

plt.tight_layout()
plt.show()

# =========================================================
# 13. SEPARATE HEATMAPS PER SOURCE (publication-quality)
# =========================================================
for src in ["GO:BP", "KEGG", "REAC", "HP"]:

    sub = enrich_top[enrich_top["source"] == src]

    if sub.empty:
        continue

    heatmap_data = sub.pivot_table(
        index="name",
        columns="Cluster",
        values="p_value",
        aggfunc="min"
    )

    heatmap_data = -np.log10(heatmap_data)

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        heatmap_data,
        cmap="viridis",
        linewidths=0.5
    )

    plt.title(f"{src} Enrichment (-log10 p)")
    plt.xlabel("Cluster")
    plt.ylabel("Term")

    plt.tight_layout()
    plt.show()

# Sample-Level Embedding: PCA, UMAP, and t-SNE

Project multi-omics data into low-dimensional spaces to visualize sample structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# ===============================
# Ensure COLORS is list
# ===============================
if isinstance(COLORS, dict):
    COLORS = list(COLORS.values())

# ===============================
# 1. Clean input (gene-level matrix)
# ===============================
X_safe = X.replace([np.inf, -np.inf], np.nan).dropna()

# ===============================
# 2. TRANSPOSE → sample-level matrix
# ===============================
X_samples = X_safe.T   # shape: (n_samples, n_features)

# ===============================
# 3. Align cancer labels
# ===============================
# cancer_labels must match samples
cancer_labels = pd.Series(cancer_labels, index=X_samples.index)

# Safety check
assert len(cancer_labels) == X_samples.shape[0], "Mismatch in labels and samples!"

# ===============================
# 4. Color mapping
# ===============================
unique_cancers = sorted(cancer_labels.unique())
color_map = {c: i for i, c in enumerate(unique_cancers)}

label_ids = cancer_labels.map(color_map).values
plot_colors = [COLORS[i % len(COLORS)] for i in label_ids]

# ===============================
# 5. Standardize
# ===============================
X_scaled = StandardScaler().fit_transform(X_samples)

# ===============================
# 6. PCA
# ===============================
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# ===============================
# 7. UMAP
# ===============================
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    random_state=42
)
X_umap = reducer.fit_transform(X_scaled)

# ===============================
# 8. t-SNE (adaptive perplexity)
# ===============================
n_samples = X_scaled.shape[0]
perplexity = min(30, max(5, n_samples // 3))

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate='auto',
    init='pca',
    random_state=42
)
X_tsne = tsne.fit_transform(X_scaled)

# ===============================
# 9. Plot panel
# ===============================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# PCA
axes[0].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=plot_colors,
    s=30,
    alpha=0.8
)
axes[0].set_title("PCA (Samples)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# UMAP
axes[1].scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=plot_colors,
    s=30,
    alpha=0.8
)
axes[1].set_title("UMAP (Samples)")
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")

# t-SNE
axes[2].scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=plot_colors,
    s=30,
    alpha=0.8
)
axes[2].set_title("t-SNE (Samples)")
axes[2].set_xlabel("t-SNE1")
axes[2].set_ylabel("t-SNE2")

# ===============================
# 10. Shared legend
# ===============================
handles = [
    plt.Line2D(
        [0], [0],
        marker='o',
        linestyle='',
        label=c,
        color=COLORS[i % len(COLORS)],
        markersize=6
    )
    for i, c in enumerate(unique_cancers)
]

fig.legend(
    handles=handles,
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    fontsize=8
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import umap

# =========================================================
# 0. OUTPUT DIRECTORY
# =========================================================
RESULT_DIR = Path("./results")
RESULT_DIR.mkdir(exist_ok=True)

# =========================================================
# 1. PREP DATA
# =========================================================
X_safe = X.replace([np.inf, -np.inf], np.nan).dropna()

gene_names = X_safe.index

print("[INFO] Clean matrix:", X_safe.shape)

# =========================================================
# 2. SCALE
# =========================================================
X_scaled = StandardScaler().fit_transform(X_safe)

# =========================================================
# 3. UMAP + CLUSTER
# =========================================================
reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.15,
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

k = 8

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_scaled)

cluster_df = pd.DataFrame({
    "Gene": gene_names,
    "Cluster": cluster_labels
})

print("[INFO] Clusters:", cluster_df.shape)

# =========================================================
# 4. COLORS
# =========================================================
if isinstance(COLORS, dict):
    COLORS_LIST = list(COLORS.values())
else:
    COLORS_LIST = COLORS

umap_colors = [
    COLORS_LIST[c % len(COLORS_LIST)]
    for c in cluster_labels
]

# =========================================================
# 5. HEATMAP DATA
# =========================================================
enrich_top = enrich_top.copy()

enrich_top["score"] = -np.log10(
    enrich_top["p_value"]
)

heatmap_data = enrich_top.pivot_table(
    index="name",
    columns="Cluster",
    values="score",
    aggfunc="max"
)

# =========================================================
# 6. MATPLOTLIB PANEL
# =========================================================
fig = plt.figure(figsize=(22, 10))

# ---------------------------------------------------------
# (A) UMAP
# ---------------------------------------------------------
ax1 = plt.subplot2grid((1, 2), (0, 0))

ax1.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=umap_colors,
    s=6,
    alpha=0.8
)

# ---------------------------------------------------------
# Cluster legend
# ---------------------------------------------------------
handles = []

for cid in sorted(np.unique(cluster_labels)):

    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label=f"Cluster {cid}",
            markerfacecolor=COLORS_LIST[cid % len(COLORS_LIST)],
            markersize=7
        )
    )

ax1.legend(
    handles=handles,
    loc="lower right",
    fontsize=10,
    frameon=True
)

ax1.set_title(
    "",
    fontsize=12
)

ax1.set_xlabel("UMAP1")
ax1.set_ylabel("UMAP2")

# ---------------------------------------------------------
# (B) HEATMAP
# ---------------------------------------------------------
ax2 = plt.subplot2grid((1, 2), (0, 1))

sns.heatmap(
    heatmap_data,
    cmap="viridis",
    ax=ax2,
    linewidths=0.3
)

ax2.set_title(
    "",
    fontsize=12
)

ax2.set_xlabel("Cluster")
ax2.set_ylabel("Pathway")

# =========================================================
# SAVE PANEL
# =========================================================
plt.tight_layout()

panel_png = RESULT_DIR / "umap_heatmap_panel.png"
panel_pdf = RESULT_DIR / "umap_heatmap_panel.pdf"
panel_svg = RESULT_DIR / "umap_heatmap_panel.svg"

plt.savefig(
    panel_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    panel_pdf,
    bbox_inches="tight"
)

plt.savefig(
    panel_svg,
    bbox_inches="tight"
)

print(f"✅ Saved PNG: {panel_png}")
print(f"✅ Saved PDF: {panel_pdf}")
print(f"✅ Saved SVG: {panel_svg}")

plt.show()

# =========================================================
# 7. SANKEY
# =========================================================

# ---------------------------------------------------------
# Limit pathways
# ---------------------------------------------------------
TOP_PATHWAYS = 25

df_sankey = enrich_top.copy()

top_pathways = (
    df_sankey.groupby("name")["score"]
    .max()
    .sort_values(ascending=False)
    .head(TOP_PATHWAYS)
    .index
)

df_sankey = df_sankey[
    df_sankey["name"].isin(top_pathways)
]

# ---------------------------------------------------------
# Nodes
# ---------------------------------------------------------
clusters = sorted(
    df_sankey["Cluster"].unique()
)

cluster_nodes = [
    f"Cluster {c}"
    for c in clusters
]

pathways = list(
    df_sankey["name"].unique()
)

all_nodes = cluster_nodes + pathways

node_idx = {
    n: i
    for i, n in enumerate(all_nodes)
}

# ---------------------------------------------------------
# Links
# ---------------------------------------------------------
sources = []
targets = []
values = []

for _, row in df_sankey.iterrows():

    c = f"Cluster {row['Cluster']}"
    p = row["name"]

    sources.append(node_idx[c])
    targets.append(node_idx[p])
    values.append(row["score"])

# ---------------------------------------------------------
# Colors
# ---------------------------------------------------------
cluster_colors = [
    COLORS_LIST[i % len(COLORS_LIST)]
    for i in range(len(cluster_nodes))
]

pathway_colors = [
    "rgba(150,150,150,0.7)"
] * len(pathways)

node_colors = (
    cluster_colors +
    pathway_colors
)

# =========================================================
# 8. BUILD SANKEY
# =========================================================
fig_sankey = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                label=all_nodes,
                color=node_colors,
                pad=12,
                thickness=12
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values
            )
        )
    ]
)

fig_sankey.update_layout(
    title_text="",
    font_size=10
)

# =========================================================
# 9. SAVE SANKEY
# =========================================================
html_path = RESULT_DIR / "cluster_pathway_sankey.html"
png_path = RESULT_DIR / "cluster_pathway_sankey.png"

fig_sankey.write_html(
    str(html_path)
)

# Requires:
# pip install kaleido
fig_sankey.write_image(
    str(png_path),
    width=1400,
    height=900,
    scale=2
)

print(f"✅ Saved HTML: {html_path}")
print(f"✅ Saved PNG: {png_path}")

fig_sankey.show()

In [ ]:
# X: gene × (cancer features)
# assume columns like: BRCA_ge, LUAD_ge, etc.

# -------------------------------
# Extract dominant cancer per gene
# -------------------------------
cancer_names = [c.split("_")[0] for c in X.columns]

X_cancer = pd.DataFrame(X.values, index=X.index, columns=cancer_names)

# aggregate per cancer
X_cancer_mean = X_cancer.groupby(level=0, axis=1).mean()

# dominant cancer
dominant_cancer = X_cancer_mean.idxmax(axis=1)

# map to colors
unique_cancers = sorted(dominant_cancer.unique())
cancer_map = {c: i for i, c in enumerate(unique_cancers)}

gene_colors = [
    COLORS[cancer_map[c] % len(COLORS)]
    for c in dominant_cancer
]

# -------------------------------
# Plot UMAP
# -------------------------------
plt.figure(figsize=(6,5))
plt.scatter(
    X_umap[:,0],
    X_umap[:,1],
    c=gene_colors,
    s=6,
    alpha=0.8
)

plt.title("Gene UMAP Colored by Dominant Cancer")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

## Multi-level biological interpretation

We construct a hierarchical network connecting clusters, genes, and pathways using a Sankey diagram. This representation captures both **membership relationships** and **functional enrichment strength**.

---

### 1. Cluster–Gene Assignment

Each cluster $C_k$ is associated with a subset of genes:

$$
\mathcal{G}_k = \{ g_1, g_2, \dots, g_n \}
\tag{1}
$$

To ensure clarity and visualization quality, we restrict the number of genes per cluster:

$$
|\mathcal{G}_k| \leq G_{\max}
\tag{2}
$$

where $G_{\max}$ is the maximum number of genes retained (e.g., 10 or 30).

---

### 2. Pathway Enrichment Scoring

For each cluster $C_k$, pathway enrichment analysis yields a set of pathways $\mathcal{P}_k$ with associated p-values.

We convert statistical significance into a score:

$$
S_p = -\log_{10}(p_p)
\tag{3}
$$

where:
- $p_p$ is the enrichment p-value of pathway $p$
- $S_p$ reflects pathway importance

We retain only the top pathways:

$$
|\mathcal{P}_k| \leq P_{\max}
\tag{4}
$$

---

### 3. Graph Construction

The Sankey diagram is defined as a directed weighted graph:

$$
\mathcal{G} = (\mathcal{V}, \mathcal{E})
\tag{5}
$$

where nodes include:

$$
\mathcal{V} = \mathcal{C} \cup \mathcal{G} \cup \mathcal{P}
\tag{6}
$$

- $\mathcal{C}$: clusters  
- $\mathcal{G}$: genes  
- $\mathcal{P}$: pathways  

---

### 4. Edge Definitions

#### (a) Cluster → Gene edges

Each gene is linked to its cluster:

$$
w(C_k \rightarrow g) = 1
\tag{7}
$$

This represents **membership** without weighting.

---

#### (b) Gene → Pathway edges

Each gene contributes to enriched pathways with strength proportional to pathway significance:

$$
w(g \rightarrow p) = S_p
\tag{8}
$$

This encodes **functional importance**.

---

### 5. Interpretation

The resulting Sankey diagram captures:

- Cluster structure (left layer)  
- Gene composition (middle layer)  
- Biological function (right layer)  

Flow thickness reflects:

- Uniform membership (Cluster → Gene)  
- Enrichment strength (Gene → Pathway)  

---

### 6. Visualization Design

To enhance interpretability:

Cluster nodes are assigned distinct colors:

$$
\text{color}(C_k) = \text{COLORS}[k]
\tag{9}
$$

Gene nodes use neutral tones:

$$
\text{color}(g) = \text{light gray}
\tag{10}
$$

Pathways use darker tones:

$$
\text{color}(p) = \text{dark gray}
\tag{11}
$$

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Use your enrichment results
# enrich_top must contain: Cluster, name, p_value

df = enrich_top.copy()

# Convert significance
df["score"] = -np.log10(df["p_value"])

# Keep top pathways globally (avoid clutter)
TOP_PATHWAYS = 20
top_pathways = (
    df.groupby("name")["score"]
    .max()
    .sort_values(ascending=False)
    .head(TOP_PATHWAYS)
    .index
)

df = df[df["name"].isin(top_pathways)]

# Left nodes: clusters
clusters = sorted(df["Cluster"].unique())
cluster_nodes = [f"Cluster {c}" for c in clusters]

# Right nodes: pathways
pathways = list(df["name"].unique())

# Combine
all_nodes = cluster_nodes + pathways

node_indices = {name: i for i, name in enumerate(all_nodes)}

sources = []
targets = []
values = []

for _, row in df.iterrows():
    c = f"Cluster {row['Cluster']}"
    p = row["name"]
    score = row["score"]

    sources.append(node_indices[c])
    targets.append(node_indices[p])
    values.append(score)



# Cluster colors
cluster_colors = [
    COLORS[i % len(COLORS)]
    for i in range(len(cluster_nodes))
]

# Pathway colors (neutral gray)
pathway_colors = ["rgba(150,150,150,0.6)"] * len(pathways)

node_colors = cluster_colors + pathway_colors





## Cluster → Gene → Pathway Sankey Construction

This visualization represents a **three-layer biological hierarchy**:

$$
\text{Cluster} \rightarrow \text{Gene} \rightarrow \text{Pathway} \tag{1}
$$

---

## 1. Data Preparation

Let:
- \( \mathcal{C} \) = set of clusters  
- \( \mathcal{G} \) = set of genes  
- \( \mathcal{P} \) = set of enriched pathways  

The enrichment score is defined as:

$$
\text{score}_p = -\log_{10}(p\text{-value}_p) \tag{2}
$$

---

## 2. Top-k Selection

Top genes per cluster:

$$
\mathcal{G}_c^{(\text{top})} = \text{Top-}k \text{ genes in cluster } c \tag{3}
$$

Top pathways per cluster:

$$
\mathcal{P}_c^{(\text{top})} = \text{Top-}k \text{ pathways ranked by } \text{score}_p \tag{4}
$$

---

## 3. Filtering by Functional Support

We retain only clusters with enrichment:

$$
\mathcal{C}^* = \left\{ c \in \mathcal{C} \mid \exists p \in \mathcal{P}_c^{(\text{top})} \right\} \tag{5}
$$

Gene set is restricted to enriched clusters:

$$
\mathcal{G}^* = \bigcup_{c \in \mathcal{C}^*} \mathcal{G}_c^{(\text{top})} \tag{6}
$$

---

## 4. Graph Construction

### Nodes

$$
\mathcal{V} = \mathcal{C}^* \cup \mathcal{G}^* \cup \mathcal{P}^* \tag{7}
$$

---

### Edges

#### Cluster → Gene

Each gene belongs to a cluster:

$$
E_{C \rightarrow G} = \{ (c, g) \mid g \in \mathcal{G}_c^{(\text{top})} \} \tag{8}
$$

Edge weights:

$$
w(c,g) = 1 \tag{9}
$$

---

#### Gene → Pathway

$$
E_{G \rightarrow P} = \{ (g, p) \mid g \in \mathcal{G}_c^{(\text{top})},\ p \in \mathcal{P}_c^{(\text{top})} \} \tag{10}
$$

Weights reflect enrichment strength:

$$
w(g,p) = -\log_{10}(p\text{-value}_p) \tag{11}
$$

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ===============================
# 1. Prepare data
# ===============================
df_clusters = cluster_df.copy()
df_enrich = enrich_top.copy()

df_enrich["score"] = -np.log10(df_enrich["p_value"])

TOP_GENES_PER_CLUSTER = 5
TOP_PATHWAYS_PER_CLUSTER = 5

# ===============================
# 2. Top genes / pathways
# ===============================
df_clusters_top = (
    df_clusters
    .groupby("Cluster")
    .head(TOP_GENES_PER_CLUSTER)
)

df_enrich_top = (
    df_enrich
    .sort_values("score")
    .groupby("Cluster")
    .tail(TOP_PATHWAYS_PER_CLUSTER)
)

# ===============================
# 🔥 3. KEEP ONLY GENES WITH PATHWAYS
# ===============================
valid_clusters = df_enrich_top["Cluster"].unique()

# genes in clusters that have enrichment
df_clusters_top = df_clusters_top[
    df_clusters_top["Cluster"].isin(valid_clusters)
]

# OPTIONAL stricter: only genes contributing to enriched clusters
genes_with_pathways = set()

for _, row in df_enrich_top.iterrows():
    c = row["Cluster"]
    genes_with_pathways.update(
        df_clusters_top[df_clusters_top["Cluster"] == c]["Gene"]
    )

df_clusters_top = df_clusters_top[
    df_clusters_top["Gene"].isin(genes_with_pathways)
]

# ===============================
# 4. Build nodes
# ===============================
clusters = sorted(df_clusters_top["Cluster"].unique())
cluster_nodes = [f"Cluster {c}" for c in clusters]

genes = df_clusters_top["Gene"].unique().tolist()
gene_nodes = [f"G:{g}" for g in genes]

pathways = df_enrich_top["name"].unique().tolist()

all_nodes = cluster_nodes + gene_nodes + pathways
node_idx = {n: i for i, n in enumerate(all_nodes)}

# ===============================
# 5. Build links
# ===============================
sources, targets, values, link_colors = [], [], [], []

gene_to_cluster = dict(
    zip(df_clusters_top["Gene"], df_clusters_top["Cluster"])
)

# ===============================
# COLORS
# ===============================
cluster_color_map = {
    c: COLORS[i % len(COLORS)]
    for i, c in enumerate(clusters)
}

def rgba(color_hex, alpha=0.4):
    color_hex = color_hex.lstrip("#")
    r, g, b = tuple(int(color_hex[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

# -------------------------------
# Cluster → Gene (COLORED PATCHES)
# -------------------------------
for _, row in df_clusters_top.iterrows():
    c = row["Cluster"]
    g = row["Gene"]

    c_node = f"Cluster {c}"
    g_node = f"G:{g}"

    sources.append(node_idx[c_node])
    targets.append(node_idx[g_node])
    values.append(1)

    # 🔥 cluster-colored links
    link_colors.append(rgba(cluster_color_map[c], 0.5))

# -------------------------------
# Gene → Pathway
# -------------------------------
for _, row in df_enrich_top.iterrows():
    c = row["Cluster"]
    p = row["name"]

    genes_in_cluster = df_clusters_top[
        df_clusters_top["Cluster"] == c
    ]["Gene"]

    for g in genes_in_cluster:
        g_node = f"G:{g}"

        if g_node not in node_idx:
            continue

        sources.append(node_idx[g_node])
        targets.append(node_idx[p])
        values.append(row["score"])

        # lighter neutral links
        link_colors.append("rgba(120,120,120,0.25)")

# ===============================
# 6. Node colors
# ===============================
cluster_colors = [cluster_color_map[c] for c in clusters]

gene_colors = [
    cluster_color_map[gene_to_cluster[g.replace("G:", "")]]
    for g in gene_nodes
]

pathway_colors = ["rgba(60,60,60,0.85)"] * len(pathways)

node_colors = cluster_colors + gene_colors + pathway_colors

# ===============================
# 7. Positioning
# ===============================
node_x = (
    [0.08] * len(cluster_nodes) +
    [0.50] * len(gene_nodes) +
    [0.92] * len(pathways)
)

node_y = (
    list(np.linspace(0.08, 0.92, len(cluster_nodes))) +
    list(np.linspace(0.05, 0.95, len(gene_nodes))) +
    list(np.linspace(0.08, 0.92, len(pathways)))
)

# ===============================
# 8. Labels
# ===============================
def shorten_label(s, max_len=45):
    return s if len(s) <= max_len else s[:max_len] + "..."

labels = [
    shorten_label(n) if not n.startswith("Cluster") and not n.startswith("G:")
    else n
    for n in all_nodes
]

# ===============================
# 9. Plot
# ===============================
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors,
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors   
    )
)])

fig.update_layout(
    title=dict(
        text="Cluster → Gene → Pathway Sankey",
        x=0.5,            # center horizontally
        xanchor='center'  # anchor at center
    ),
    font_size=16,
    width=1200,
    height=1100,
    margin=dict(l=10, r=10, t=70, b=60)
)

fig.show()

## Pathway Enrichment Filtering and Visualization

Refine enrichment results to retain only statistically meaningful pathways and visualize the strongest biological signals across clusters.


In [ ]:
# =========================================================
# Keep significant pathways
# =========================================================
enrich_df = enrich_df[
    enrich_df["p_value"] < 0.05
].copy()

# =========================================================
# Top N pathways per cluster
# =========================================================
TOP_N = 5

enrich_top = (
    enrich_df
    .sort_values("p_value")
    .groupby("Cluster")
    .head(TOP_N)
    .copy()
)

# =========================================================
# Convert p-values
# =========================================================
enrich_top["-log10(p)"] = -np.log10(
    enrich_top["p_value"]
)

from matplotlib.lines import Line2D
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# PLOT
# =========================================================

plt.figure(figsize=(10, 8))

ax = sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    sizes=(50, 300)
)

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Cluster Pathway Enrichment",
    fontsize=20,
    pad=30
)

# =========================================================
# REMOVE AXIS TITLES
# =========================================================

ax.set_xlabel("")
ax.set_ylabel("")

# =========================================================
# TICKS
# =========================================================

ax.tick_params(
    axis="x",
    labelsize=14,
    length=5,
    width=1
)

ax.tick_params(
    axis="y",
    labelsize=12,
    length=5,
    width=1
)

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------
# STYLE
# ---------------------------------------------------------

sns.set_style("whitegrid")

fig, ax = plt.subplots(
    figsize=(10, 8)
)

# ---------------------------------------------------------
# DOTPLOT
# ---------------------------------------------------------

scatter = sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    sizes=(40, 300),
    alpha=0.9,
    edgecolor="black",
    linewidth=0.4,
    ax=ax
)

# ---------------------------------------------------------
# TITLES
# ---------------------------------------------------------

# ax.set_title(
#     "Cluster Pathway Enrichment",
#     fontsize=22,
#     pad=20
# )

ax.set_xlabel("")
ax.set_ylabel("")

# ---------------------------------------------------------
# TICKS
# ---------------------------------------------------------

ax.tick_params(
    axis="x",
    labelsize=14,
    length=4,
    width=1
)

ax.tick_params(
    axis="y",
    labelsize=11,
    length=3,
    width=1
)

# # ---------------------------------------------------------
# # SPINES
# # ---------------------------------------------------------

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# # ---------------------------------------------------------
# # LIGHT GRID
# # ---------------------------------------------------------

# ax.grid(
#     axis="x",
#     linestyle="--",
#     alpha=0.25
# )

# ---------------------------------------------------------
# REMOVE DEFAULT LEGEND
# ---------------------------------------------------------

if ax.legend_:
    ax.legend_.remove()

# ---------------------------------------------------------
# SOURCE LEGEND
# ---------------------------------------------------------

source_handles = [

    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor=sns.color_palette()[0],
        markeredgecolor="black",
        markersize=8,
        label="GO:BP"
    ),

    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor=sns.color_palette()[1],
        markeredgecolor="black",
        markersize=8,
        label="REAC"
    ),

    Line2D(
        [0], [0],
        marker="o",
        color="w",
        markerfacecolor=sns.color_palette()[2],
        markeredgecolor="black",
        markersize=8,
        label="KEGG"
    )
]

# ---------------------------------------------------------
# SIZE LEGEND (HOLLOW)
# ---------------------------------------------------------

size_values = [5, 10, 15, 20, 25, 30]

size_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markersize=np.sqrt(v * 8),
        label=str(v)
    )
    for v in size_values
]

# ---------------------------------------------------------
# MERGED LEGEND
# ---------------------------------------------------------

legend = ax.legend(
    handles=source_handles + size_handles,
    labels=["GO:BP", "REAC", "KEGG"] + [str(v) for v in size_values],
    title="source\n\n-log10(p)",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    fontsize=10,
    title_fontsize=11,
    frameon=True,
    ncol=1
)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

plt.tight_layout()

plt.savefig(
    "cluster_pathway_dotplot.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# # =========================================================
# # LEAVE ROOM FOR LEGENDS
# # =========================================================

# plt.tight_layout(rect=[0, 0, 0.82, 1])

# # =========================================================
# # LAYOUT
# # =========================================================

# plt.tight_layout(rect=[0, 0.08, 1, 1])

# plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    sizes=(50, 300)
)

# # ---------------------------------------------------------
# # Title farther from plot
# # ---------------------------------------------------------

# plt.title(
#     "Cluster Pathway Enrichment",
#     fontsize=20,
#     pad=25          # increase spacing
# )

# ---------------------------------------------------------
# Remove axis titles
# ---------------------------------------------------------

plt.xlabel("")
plt.ylabel("")

# Alternative:
# plt.gca().set_xlabel(None)
# plt.gca().set_ylabel(None)

# ---------------------------------------------------------
# Tick styling
# ---------------------------------------------------------

plt.xticks(fontsize=14)
plt.yticks(fontsize=12)

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(10, 8))

sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    sizes=(50, 300),
    ax=ax
)

# ---------------------------------------------------------
# REMOVE GRID
# ---------------------------------------------------------

ax.grid(False)

# ---------------------------------------------------------
# REMOVE AXIS LABELS
# ---------------------------------------------------------

ax.set_xlabel("")
ax.set_ylabel("")

# ---------------------------------------------------------
# TICKS
# ---------------------------------------------------------

ax.tick_params(
    axis="x",
    labelsize=14,
    length=4,
    width=1
)

ax.tick_params(
    axis="y",
    labelsize=12,
    length=3,
    width=1
)

# ---------------------------------------------------------
# REMOVE DEFAULT LEGEND
# ---------------------------------------------------------

if ax.legend_:
    ax.legend_.remove()

# ---------------------------------------------------------
# SOURCE LEGEND
# ---------------------------------------------------------

source_handles = [

    Line2D(
        [0],[0],
        marker="o",
        linestyle="",
        markerfacecolor=sns.color_palette()[0],
        markeredgecolor="black",
        markersize=8,
        label="GO:BP"
    ),

    Line2D(
        [0],[0],
        marker="o",
        linestyle="",
        markerfacecolor=sns.color_palette()[1],
        markeredgecolor="black",
        markersize=8,
        label="REAC"
    ),

    Line2D(
        [0],[0],
        marker="o",
        linestyle="",
        markerfacecolor=sns.color_palette()[2],
        markeredgecolor="black",
        markersize=8,
        label="KEGG"
    )
]

# ---------------------------------------------------------
# SIZE LEGEND (HOLLOW)
# ---------------------------------------------------------

size_values = [5, 10, 15, 20, 25, 30]

size_handles = [

    Line2D(
        [0],[0],
        marker="o",
        linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=1.2,
        markersize=np.sqrt(v * 8),
        label=str(v)
    )

    for v in size_values
]

# ---------------------------------------------------------
# COMBINED LEGEND
# ---------------------------------------------------------

legend = ax.legend(
    handles=source_handles + size_handles,
    labels=["GO:BP", "REAC", "KEGG"] + [str(v) for v in size_values],
    title="source\n-log10(p)",
    loc="upper right",          # inside upper-right corner
    bbox_to_anchor=(0.99, 0.99),
    fontsize=10,
    title_fontsize=11,
    frameon=True,
    ncol=1,
    borderpad=0.6
)

# # ---------------------------------------------------------
# # SPINES
# # ---------------------------------------------------------

# # ax.spines["top"].set_visible(False)
# # ax.spines["right"].set_visible(False)

# # ---------------------------------------------------------
# # LEAVE ROOM FOR LEGEND
# # ---------------------------------------------------------

# fig.subplots_adjust(bottom=0.15)

plt.show()

plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# =====================================================
# FIGURE
# =====================================================

fig, ax = plt.subplots(
    figsize=(8, 8)
)

# =====================================================
# COLORS
# =====================================================

palette = {
    "GO:BP":"#1f77b4",
    "REAC":"#ff7f0e",
    "KEGG":"#2ca02c"
}

# =====================================================
# HOLLOW SCATTERPLOT
# =====================================================

sns.scatterplot(
    data=enrich_top,
    x="Cluster",
    y="name",
    size="-log10(p)",
    hue="source",
    sizes=(40,300),

    palette=palette,

    facecolors="none",          # hollow circles
    linewidth=1.5,
    edgecolor="black",

    legend=False,
    ax=ax
)

# =====================================================
# REMOVE GRID
# =====================================================

ax.grid(False)

# =====================================================
# REMOVE AXIS LABELS
# =====================================================

ax.set_xlabel("")
ax.set_ylabel("")

# =====================================================
# TICKS
# =====================================================

ax.tick_params(
    axis="x",
    labelsize=14,
    length=4,
    width=1,
    direction="out"
)

ax.tick_params(
    axis="y",
    labelsize=12,
    length=3,
    width=1,
    direction="out"
)

# =====================================================
# CLEAN SPINES
# =====================================================

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# =====================================================
# COMBINED LEGEND
# =====================================================

# section headers
header_source = Line2D(
    [],[],
    linestyle="",
    label="Source"
)

header_size = Line2D(
    [],[],
    linestyle="",
    label="-log10(p)"
)

legend_handles = (
    [header_source]
    + source_handles
    + [header_size]
    + size_handles
)

legend_labels = (
    ["Source"]
    + ["GO:BP","REAC","KEGG"]
    + ["-log10(p)"]
    + [str(v) for v in size_values]
)

legend = ax.legend(

    handles=legend_handles,
    labels=legend_labels,

    loc="lower left",

    # lower-right outside plot
    bbox_to_anchor=(1.02,0.02),

    fontsize=10,
    frameon=False,

    borderpad=0.2,
    labelspacing=0.5,
    handletextpad=0.8,
    handlelength=1.0
)

# =====================================================



# =====================================================
# FORMAT HEADERS
# =====================================================

for txt, handle in zip(
    legend.get_texts(),
    legend.legend_handles
):

    if txt.get_text() in ["Source", "-log10(p)"]:

        txt.set_fontsize(12)
        # txt.set_fontweight("bold")

        handle.set_visible(False)

# =====================================================
# MAKE ROOM FOR LEGEND
# =====================================================

plt.tight_layout(
    rect=[0,0,0.84,1]
)

# =====================================================
# LAYOUT
# =====================================================

plt.tight_layout()

plt.show()